<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/%EA%B8%B0%EB%B3%B8%20%ED%94%84%EB%A1%9C%ED%86%A0%ED%83%80%EC%9E%85(%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%EC%9C%84%EC%A3%BC%20%EC%8B%9C%EC%8A%A4%ED%85%9C%20%EA%B5%AC%EC%B6%95%20RAG%EC%95%84%EC%A7%81%20%EB%AF%B8%EA%B5%AC%ED%98%84).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 「잠긴 종(The Drowned Bell)」 — 팀 공유용 코드 보기 노트북

`team_share_game.py`의 코드를 **구역별 셀**로 나눠 보기 쉽게 만든 노트북입니다. 왼쪽 **목차(≡ 아이콘)**에서 구역으로 바로 이동할 수 있습니다.

- 코드 내용은 `team_share_game.py`와 같습니다 (바꾼 것: 아래 '노트북으로 옮기며 바꾼 것').
- **보기만 할 때는 실행하지 않아도 됩니다.**
- 실행하려면 같은 폴더에 `game/data/*.json` 9개가 있어야 합니다 (`share.zip`을 풀면 됨). 위에서부터 차례로 실행하세요.

### 노트북으로 옮기며 바꾼 것
- `PROJECT_DIR = Path(__file__)…` → `Path.cwd()` (노트북에는 파일 위치가 없어서 현재 폴더 기준)
- 맨 끝 명령줄 실행 메뉴 호출(`if __name__ == "__main__"`)은 넣지 않았습니다. 대신 마지막에 사용 예시 셀이 있습니다.

## 0. 파일 설명 (원본 머리말)

```text
「잠긴 종(The Drowned Bell)」 — 팀 공유용 단일 코드 파일 (2026-11-05, W7 완료 시점)

이 파일은 프로젝트의 Python 코드 11개 파일을 순서대로 이어 붙인 '읽기·공유용' 묶음이다.
코드 내용은 원본과 같고, 바꾼 것은 파일을 합치기 위해 필요한 부분뿐이다 (아래 "통합하며 바꾼 것").
원본 프로젝트(game/, server/, tools/)가 기준이며, 이 파일을 고쳐도 원본에는 반영되지 않는다.

────────────────────────────────────────────────────────────────────────────────
게임 개요
  SRD 5.2 기반 시나리오 「잠긴 종」을 1인 1캐릭터, 선택지 기반 텍스트 어드벤처로 만든 MVP.
  - 장면 1(에코 타운)과 장면 2(마른 우물과 수직 갱도)를 끝까지 플레이할 수 있다. 장면 3~6은 미구현.
  - 주사위·판정·피해·상태는 모두 Python 코드(Engine)가 정한다.
  - AI(Qwen 등) 서술은 선택 기능이며 기본은 꺼져 있다. 실제 모델로는 아직 검증하지 않았다.
  - RAG·임베딩·벡터 DB(Chroma)는 쓰지 않는다. 정보 공개는 카드 메타데이터 조건으로 정한다.

────────────────────────────────────────────────────────────────────────────────
반드시 필요한 외부 데이터 (이 파일과 같은 폴더 기준)
  game/data/cards.json  scene1.json  scene2.json  characters.json  items.json
            traps.json  monsters.json  encounters.json  rules.json
    → 없으면 Content()에서 FileNotFoundError. (scenario.jsonl은 게임이 읽지 않음)
  web/  (index.html, app.js, audio.js, style.css, assets/*.svg)
    → 웹 화면(web 명령)에만 필요. 엔진·터미널 화면·테스트에는 필요 없다.
  외부 라이브러리: 없음 (Python 3.10 이상 표준 라이브러리만).
    Ollama 서술(--narrator ollama)을 쓰려면 Ollama 서버가 따로 실행 중이어야 한다.

────────────────────────────────────────────────────────────────────────────────
실행 예시 (이 파일을 프로젝트 폴더 맨 위, game/ 와 web/ 옆에 둔다)
  python team_share_game.py check                # 필요한 데이터 파일이 있는지 확인
  python team_share_game.py demo                 # 입력 없이 장면 1 앞부분을 자동 진행해 출력
  python team_share_game.py cli                  # 터미널에서 직접 플레이 (캐릭터 선택부터)
  python team_share_game.py cli --char harin --seed 7
  python team_share_game.py cli --combat rats    # 전투만 시험
  python team_share_game.py web --port 8000      # 브라우저 게임 → http://127.0.0.1:8000
  python team_share_game.py sim --runs 50 --enc rats crabs   # 전투 난이도 측정
  python team_share_game.py playscript           # 정해진 경로로 장면 1 자동 진행
  (Python에서) from team_share_game import Engine; e = Engine(seed=7, char_id="harin")

────────────────────────────────────────────────────────────────────────────────
게임 한 번의 행동이 처리되는 흐름
  1. Engine(char_id=…) 생성
       Content()가 game/data/*.json을 읽음 → Character 1명 → GameState → 첫 방에 _enter → 체크포인트
  2. engine.available_choices()  지금 고를 수 있는 선택지(ChoiceView) 목록
  3. engine.choose(선택지 id)     id 접두어로 처리 함수를 고름
       장면 선택지 → _run_choice: before 효과 → roll_check(판정) → success/failure 효과
       cmb:… → combat_choose(전투) / move: → _enter / buy: → _buy / use: → _use_item …
  4. _apply(효과 목록)            GameState(flag·소지품·HP·카드 공개·이동·전투…)를 바꿈
  5. Event 목록을 돌려줌          터미널(cli)·웹 서버(Game.reply)·AI 서술기가 이것을 보여 줌
  6. 다시 2번으로. state.ended가 "game_over"면 restart_from_checkpoint(), "scene2_complete"면 끝.
  웹: 브라우저(web/app.js) → POST /api/choose → Game.choose → Engine.choose → Game.reply(JSON) → 화면

────────────────────────────────────────────────────────────────────────────────
파일 구성 (구역 번호 = 아래 ##### 머리말)
  [1] Character, GameState      게임 중 바뀌는 값 (원본 game/model.py)
  [2] Dice, FixedDice           주사위 (game/dice.py)
  [3] Event, ChoiceView         엔진이 내보내는 결과 형식 (game/events.py)
  [4] Content                   JSON 데이터 읽기 (game/content.py)
  [5] CombatMixin               전투 (game/combat.py) — [6] Engine이 상속
  [6] Engine                    게임 엔진 핵심 (game/engine.py)
  [7] Narrator 3종              AI 서술 (game/narrator.py)
  [8] play, cli_main            터미널 화면 (game/cli.py)
  [9] Game, Session, web_main   웹 서버 (server/webapp.py)
  [10] sim_main                 전투 측정 도구 (tools/combat_sim.py)
  [11] play_script_main         자동 진행 도구 (tools/play_script.py)
  맨 끝  실행 메뉴 (공유용으로 추가)
  ※ [5]가 [6]보다 앞에 있는 이유: Engine(CombatMixin)이 상속하므로 먼저 정의되어야 한다.

패키지 설명 (원본 __init__.py):
#   「잠긴 종」 TRPG MVP 게임 엔진.
#   
#   역할 분리 (analysis/05_시스템설계/05-1a 4장, 05-2a 3장):
#     - Python(이 패키지): 선택지, 주사위, 판정, 상태·flag 변경, 카드 열람/표시 결정
#     - Qwen(narrator): 표시된 정보와 판정 결과를 문장으로 묘사만 한다
#   웹 UI용 서버. 게임 규칙은 game/ 엔진에만 있다.

────────────────────────────────────────────────────────────────────────────────
통합하며 바꾼 것 (이것 외에는 원본 코드 줄 그대로)
  - 파일 사이 import(from .engine import … 등)와 sys.path 조작 줄을 지웠다 (한 파일이라 필요 없음).
  - 표준 라이브러리 import는 맨 위에 한 번만 모았다.
  - 이름이 겹치는 main() 4개 → cli_main / web_main / sim_main / play_script_main
  - 데이터 경로 DATA, 화면 경로 WEB → 이 파일이 있는 폴더(PROJECT_DIR) 기준
  - play_script_main 안의 cli.play(…) → play(…)
  - 각 파일의 `if __name__ == "__main__":` 실행 부분 → 맨 끝 실행 메뉴 하나로
  - 원본 파일의 맨 위 설명(docstring)은 각 구역 머리말 주석으로 옮겼다.
포함하지 않은 원본 파일 (프로젝트에는 그대로 있음)
  - tools/build_cards.py : 개발용 데이터 생성기. 실행하면 game/data/cards.json을 덮어쓰고,
                           공유하지 않는 원본(시나리오 생성 스크립트·PDF)과 pypdf가 필요하다.
  - colab/make_bundle.py : 여러 파일을 zip으로 묶는 도구. 단일 파일에서는 필요 없다.
  - tests/*.py           : 테스트는 원본 폴더 구조(game.engine 등)를 import한다. 프로젝트에서 실행할 것.
```

## 표준 라이브러리 import와 기준 폴더
외부 라이브러리는 쓰지 않습니다. `PROJECT_DIR`은 데이터(`game/data`)와 화면(`web`)을 찾는 기준 폴더입니다.

In [ ]:
import argparse
import copy
import json
import math
import random
import re
import secrets
import sys
import threading
import urllib.error
import urllib.request
from collections import Counter
from dataclasses import dataclass, field
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlparse

# 이 노트북이 있는 폴더(현재 폴더). 데이터(game/data)와 화면(web)을 이 폴더 기준으로 찾는다.
# [노트북] 원래 .py 파일에서는 Path(__file__).resolve().parent
PROJECT_DIR = Path.cwd()

## [1] 캐릭터와 게임 상태 (Character, GameState)
원본 파일: `game/model.py`

```text
캐릭터와 게임 상태 (런타임). 설계: analysis/05_시스템설계/05-2 7장.
여기 있는 값은 게임 중에 바뀌는 값이다. 바꾸는 쪽은 engine.py·combat.py뿐이고,
서버(server/webapp.py)와 CLI는 읽어서 화면용으로 정리만 한다.
체크포인트는 GameState 전체를 deepcopy한 것이므로, 새 상태값은 반드시 GameState 안에 두어야 되돌리기에 포함된다.
```

**들어 있는 것**
- **`ability_mod`** (함수) — SRD 5.2 p.21 Ability Scores and Modifiers 표와 같은 값.
- **`Character`** (클래스) — 플레이 캐릭터 1명. characters.json의 항목 하나가 그대로 이 필드들로 들어온다 (Content.new_party).
- **`GameState`** (클래스) — 게임 한 판의 바뀌는 상태 전부. Engine.s로 접근한다.

In [ ]:
ABILITIES = ["str", "dex", "con", "int", "wis", "cha"]


ABILITY_KO = {"str": "근력", "dex": "민첩", "con": "건강", "int": "지능", "wis": "지혜", "cha": "매력"}


def ability_mod(score):
    """SRD 5.2 p.21 Ability Scores and Modifiers 표와 같은 값."""
    return (score - 10) // 2

In [ ]:
@dataclass
class Character:
    """플레이 캐릭터 1명. characters.json의 항목 하나가 그대로 이 필드들로 들어온다 (Content.new_party).

    conditions(상태)의 값 형식은 상태마다 다르다:
      정수 = 남은 시간(분, _pass_time이 줄임) · None = 무기한 · dict = 추가 정보
      (dying: {"success", "fail"} 죽음 구원 횟수, grappled: {몬스터 id: {"dc", "restrained"}}) · "grapple" = 붙잡혀서 생긴 Restrained
    """
    id: str
    name: str
    cls: str
    cls_ko: str
    species: str
    background: str
    level: int
    abilities: dict
    hp_max: int
    save_profs: list
    skill_profs: list
    tool_profs: list
    traits: list
    tools: list
    darkvision: int = 0
    proficiency_bonus: int = 2
    hp: int = None
    conditions: dict = field(default_factory=dict)  # 이름 -> 남은 시간(분), None이면 무기한
    hit_die: int = 8                                # Hit Point Die 면 수 [SRD 클래스 표]
    hd_left: int = None                             # 남은 Hit Point Dice (최대 = 레벨) [SRD Rules Glossary]
    heroic_inspiration: bool = False                # [SRD Rules Glossary: Heroic Inspiration]
    # --- 전투 (W6) ---
    armor: str = "none"
    shield: bool = False
    fighting_style: str = ""
    weapons: list = field(default_factory=list)
    weapon_profs: list = field(default_factory=list)
    spellcasting: str = ""                          # 주문 시전 능력치 (int / wis)
    cantrips: list = field(default_factory=list)
    spells: list = field(default_factory=list)
    features: list = field(default_factory=list)
    resources: dict = field(default_factory=dict)   # 남은 사용 횟수: slots, second_wind, action_surge 등
    armor_penalty: int = 0                          # Gray Ooze 부식 [SRD p.290]
    weapon_penalty: dict = field(default_factory=dict)
    ability_loss: dict = field(default_factory=dict)  # Shadow 근력 흡수 등. 긴 휴식에서 복구 [SRD Long Rest]
    hp_max_loss: int = 0                            # Specter 생명 흡수
    concentration: str = ""                         # 집중 중인 주문 id

    def __post_init__(self):
        if self.hp is None:
            self.hp = self.hp_max
        if self.hd_left is None:
            self.hd_left = self.hd_max

    @property
    def hd_max(self):
        return self.level

    def mod(self, ability):
        """능력 수정치 (판정·공격·구원에 더하는 값). 줄어든 능력치(ability_loss)를 반영한다."""
        return ability_mod(self.score(ability))

    def score(self, ability):
        """현재 능력치 = 원래 값 − 흡수로 줄어든 값."""
        return self.abilities[ability] - self.ability_loss.get(ability, 0)

    @property
    def max_hp(self):
        """현재 최대 HP (생명 흡수로 줄어든 값). hp_max는 원래 값."""
        return max(0, self.hp_max - self.hp_max_loss)

    @property
    def down(self):
        return self.hp <= 0

    def can_act(self):
        return not self.down

In [ ]:
@dataclass
class GameState:
    """게임 한 판의 바뀌는 상태 전부. Engine.s로 접근한다.

    flags는 원본 시나리오의 선택 flag(카드 req_flag와 연결), vars는 그 밖의 진행 상태
    (예: rope_anchored, shrieker_alerted, combat:rats). 장면 JSON의 {"flag": …}/{"var": …} 효과가 채운다.
    """
    party: list
    room: str
    scene: int = 1
    flags: set = field(default_factory=set)
    vars: dict = field(default_factory=dict)          # flag 표에 없는 상태값 (05-2 7-3)
    inventory: dict = field(default_factory=dict)     # 파티 공동 소지품: item_id -> 개수
    gold_cp: int = 0                                  # 돈은 cp 단위로 저장
    visited: set = field(default_factory=set)
    shown: dict = field(default_factory=dict)         # card_id -> 알게 된 캐릭터 id 목록 (플레이어 축)
    failed: set = field(default_factory=set)          # (choice_id, char_id): M3 재시도 규칙
    done: set = field(default_factory=set)            # 한 번만 하는 선택지 중 끝난 것
    minutes: int = 0                                  # 경과 시간
    ended: str = ""                                   # "scene1_complete" / "game_over" 등
    stock: dict = field(default_factory=dict)         # 상점 재고 (게임 중 바뀌는 값)
    deferred: dict = field(default_factory=dict)      # 나중에 공개할 카드·문장: key -> {"room", "card", "part"}
    once_done: set = field(default_factory=set)       # 한 번만 일어나는 방 진입·구매 반응
    traps: dict = field(default_factory=dict)         # trap_id -> {"count": 발동 횟수, "disarmed": bool} (W5)
    pending: dict = None                              # 플레이어 결정을 기다리는 일: 영감 재굴림 / 짧은 휴식 주사위 (W5·W6)
    combat: dict = None                               # 진행 중인 전투 (W6, game/combat.py)
    log: list = field(default_factory=list)

    def member(self, char_id):
        for c in self.party:
            if c.id == char_id:
                return c
        raise KeyError(char_id)

    def has_item(self, item_id):
        return self.inventory.get(item_id, 0) > 0

## [2] 주사위 (Dice, FixedDice)
원본 파일: `game/dice.py`

```text
주사위. 시드를 고정하거나 정해진 값을 넣어 테스트할 수 있다.
모든 무작위 값은 Dice.die() 하나를 거친다. 그래서 FixedDice가 die()만 바꿔도
d20·피해·낙하·우선권까지 모두 정해진 값으로 굴러간다 (tests/ 의 대부분이 이 방식).
```

**들어 있는 것**
- **`Dice`** (클래스) — seed가 같으면 같은 순서로 같은 값이 나온다 (화면의 '주사위 시드' 입력이 여기로 온다).
- **`FixedDice`** (클래스) — 테스트용: 미리 정한 값을 차례로 돌려준다 (d20, 피해 주사위 모두).

In [ ]:
DICE_EXPR = re.compile(r"^\s*(\d*)d(\d+)\s*([+-]\s*\d+)?\s*$")

In [ ]:
class Dice:
    """seed가 같으면 같은 순서로 같은 값이 나온다 (화면의 '주사위 시드' 입력이 여기로 온다)."""

    def __init__(self, seed=None):
        self.rng = random.Random(seed)

    def die(self, sides):
        return self.rng.randint(1, sides)

    def d20(self, mode="normal"):
        """mode: normal / advantage / disadvantage. (선택된 값, 굴린 값 목록)을 돌려준다."""
        if mode == "normal":
            r = self.die(20)
            return r, [r]
        rolls = [self.die(20), self.die(20)]
        return (max(rolls) if mode == "advantage" else min(rolls)), rolls

    def roll(self, expr):
        """'1d10', '2d6+3' 같은 식. (합계, 개별 값)을 돌려준다."""
        m = DICE_EXPR.match(expr)
        if not m:
            raise ValueError(f"주사위 식을 읽을 수 없음: {expr}")
        count = int(m.group(1) or 1)
        sides = int(m.group(2))
        bonus = int(m.group(3).replace(" ", "")) if m.group(3) else 0
        rolls = [self.die(sides) for _ in range(count)]
        return sum(rolls) + bonus, rolls

In [ ]:
class FixedDice(Dice):
    """테스트용: 미리 정한 값을 차례로 돌려준다 (d20, 피해 주사위 모두)."""

    def __init__(self, values):
        super().__init__(0)
        self.values = list(values)

    def die(self, sides):
        if not self.values:
            raise RuntimeError("FixedDice 값이 바닥남")
        v = self.values.pop(0)
        assert 1 <= v <= sides, (v, sides)
        return v

## [3] 엔진 출력 형식 (Event, ChoiceView)
원본 파일: `game/events.py`

```text
엔진이 화면·서술기(AI)에 넘기는 결과와 선택지 형식.
엔진은 화면을 직접 그리지 않는다. 결과는 Event 목록, 고를 수 있는 행동은 ChoiceView 목록으로만 내보내고,
CLI(game/cli.py)·웹 서버(server/webapp.py)·서술기(game/narrator.py)가 각자 이 형식을 읽어 보여 준다.
```

**들어 있는 것**
- **`Event`** (클래스) — 일어난 일 하나. text는 화면에 그대로 보여 줄 문장, data는 화면 연출용 값 (주사위 숫자, 카드 id 등).
- **`ChoiceView`** (클래스) — 지금 고를 수 있는 행동 하나. 화면은 label과 key를 보여 주고, 고르면 id를 Engine.choose에 넘긴다.

In [ ]:
@dataclass
class Event:
    """일어난 일 하나. text는 화면에 그대로 보여 줄 문장, data는 화면 연출용 값 (주사위 숫자, 카드 id 등)."""
    kind: str                  # card / say / check / damage / item / gold / flag / move / rest / end / status / combat
    text: str
    data: dict = field(default_factory=dict)

In [ ]:
@dataclass
class ChoiceView:
    """지금 고를 수 있는 행동 하나. 화면은 label과 key를 보여 주고, 고르면 id를 Engine.choose에 넘긴다."""
    id: str
    label: str
    actors: list               # 이 선택지를 할 수 있는 캐릭터 id (파티 행동이면 [])
    recommended: str = ""      # 보완 A: 판정에 가장 유리한 캐릭터
    check_text: str = ""
    key: str = ""              # 화면에 보여 줄 고정 키 (턴이 바뀌어도 같은 행동은 같은 키)

## [4] 데이터 불러오기 (Content)
원본 파일: `game/content.py`

```text
게임 데이터 불러오기. 데이터 생성은 tools/build_cards.py (카드), 나머지는 game/data/*.json.
장면 파일: game/data/scene*.json 을 모두 읽어 하나로 합친다 (W3).
  - 파일마다 "scene"(번호), "title", "rooms", "choices"를 가진다.
  - 장면 파일을 추가하면 엔진 수정 없이 방·선택지가 연결된다.
  - 방 ID와 선택지 ID는 모든 장면에서 유일해야 한다 (불러올 때 검사).
불러온 결과(Content 속성)를 쓰는 곳: engine.py·combat.py는 self.c.xxx로, 서버는 Game.content로 읽는다.
JSON 파일과 속성의 대응:
  cards.json → cards, room_names · scene*.json → scenes, rooms, choices, card_views, buy_reactions, effect_sets
  items.json → items, shop_order, use_order · traps.json → traps · monsters.json → monsters
  encounters.json → encounters · rules.json → rules · characters.json → _chars (new_party로 Character 생성)
```

**들어 있는 것**
- **`_load`** (함수) — (설명 없음)
- **`_load_path`** (함수) — (설명 없음)
- **`Content`** (클래스) — 읽기 전용 게임 데이터. 게임 중에 바뀌는 값(상점 재고 등)은 GameState에 둔다.

In [ ]:
DATA = PROJECT_DIR / "game" / "data"   # [통합] 원래: game/ 폴더 기준. 지금: 이 파일이 있는 폴더의 game/data


def _load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def _load_path(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

In [ ]:
class Content:
    """읽기 전용 게임 데이터. 게임 중에 바뀌는 값(상점 재고 등)은 GameState에 둔다."""

    def __init__(self, scene_paths=None):
        """scene_paths를 주면 그 장면 파일만 읽는다 (테스트에서 장면 추가를 흉내 낼 때). 기본은 data/scene*.json 전부."""
        cards = _load("cards.json")
        self.cards = {c["id"]: c for c in cards["cards"]}
        self.room_names = cards["rooms"]
        paths = scene_paths or sorted(DATA.glob("scene*.json"))
        self.scenes = sorted((_load_path(p) for p in paths), key=lambda s: s["scene"])
        self.rooms, self.choices, self.card_views, self.buy_reactions, self.effect_sets = {}, {}, {}, {}, {}
        self._room_choices = {}
        self.scene_titles, self.scene_status = {}, {}
        for sc in self.scenes:
            n = sc["scene"]
            self.scene_titles[n] = sc.get("title", f"장면 {n}")
            self.scene_status[n] = sc.get("status", "playable")
            for rid, room in sc["rooms"].items():
                if rid in self.rooms:
                    raise ValueError(f"방 ID 중복: {rid}")
                if room["scene"] != n:
                    raise ValueError(f"방 {rid}의 장면 번호({room['scene']})가 파일({n})과 다름")
                self.rooms[rid] = room
            for ch in sc.get("choices", []):
                if ch["id"] in self.choices:
                    raise ValueError(f"선택지 ID 중복: {ch['id']}")
                self.choices[ch["id"]] = ch
                self._room_choices.setdefault(ch["room"], []).append(ch)
            self.card_views.update({k: v for k, v in sc.get("card_views", {}).items() if not k.startswith("_")})
            self.buy_reactions.update({k: v for k, v in sc.get("buy_reactions", {}).items() if not k.startswith("_")})
            # "…_effects"로 끝나는 최상위 목록은 선택지에서 {"use": 이름}으로 불러 쓰는 효과 묶음
            self.effect_sets.update({k: v for k, v in sc.items() if k.endswith("_effects") and isinstance(v, list)})
        first = self.scenes[0]
        self.start_scene = first["scene"]
        self.start_room = first["start_room"]
        items = _load("items.json")
        self.items = items["items"]
        self.shop_order = items["shop_order"]
        self.use_order = [i for i in self.shop_order if "use" in self.items[i]]  # 사용 선택지의 고정 키 순서 (W5)
        self.traps = _load("traps.json")["traps"]
        self.monsters = _load("monsters.json")["monsters"]        # W6
        self.encounters = _load("encounters.json")["encounters"]
        self.rules = _load("rules.json")
        self._check_trap_refs()
        self._chars = _load("characters.json")["characters"]

    def _check_trap_refs(self):
        """장면 데이터가 부르는 함정 ID가 traps.json에 있는지 불러올 때 검사."""
        def walk(x):
            if isinstance(x, dict):
                for k in ("trap", "disarm_trap", "trap_active"):
                    if k in x and x[k] not in self.traps:
                        raise ValueError(f"없는 함정 ID: {x[k]}")
                if "combat" in x and isinstance(x["combat"], str) and x["combat"] not in self.encounters:
                    raise ValueError(f"없는 전투 ID: {x['combat']}")
                for v in x.values():
                    walk(v)
            elif isinstance(x, list):
                for v in x:
                    walk(v)
        walk(self.scenes)

    def choices_in_room(self, room_id):
        """그 방의 선택지를 데이터 순서대로 (고정 키의 기준)."""
        return self._room_choices.get(room_id, [])

    def exits(self, room_id):
        """불러온 장면에 있는 방으로만 이동 가능 (아직 없는 장면의 방은 숨김)."""
        return [r for r in self.rooms[room_id]["exits"] if r in self.rooms]

    def characters(self):
        """선택 가능한 사전 제작 캐릭터 목록 [(id, 이름, 직업)]."""
        return [(c["id"], c["name"], c["cls_ko"]) for c in self._chars]

    def new_party(self, char_id):
        """1인 1캐릭터 (05-6 ①): 선택한 캐릭터 한 명만 만든다."""
        for c in self._chars:
            if c["id"] == char_id:
                return [Character(**c)]
        raise ValueError(f"없는 캐릭터: {char_id} (가능: {[c['id'] for c in self._chars]})")

    def room_name(self, room_id):
        """방 이름. 장면 파일에 있으면 그 이름, 아직 장면 파일이 없는 방은 cards.json의 방 이름."""
        r = self.rooms.get(room_id)
        return r["name"] if r else self.room_names.get(room_id, room_id)

## [5] 전투 시스템 (CombatMixin)
원본 파일: `game/combat.py`

```text
전투 (W6). 위치 없는 1인 전투 [05-1a 보완 C]. Engine이 이 클래스를 상속한다.
주사위·명중·피해·HP·사망은 모두 이 코드가 정한다. AI 서술기는 결과 문장만 받는다.
규칙 근거 (SRD 5.2, 실제로 읽은 쪽):
  전투 순서·우선권·행동 p.13-14, 공격 굴림·치명타·피해 p.6-7·15-16, 저항·취약·면역 p.17,
  0 HP·죽음 구원 p.17-18, 상태 용어집(Grappled·Restrained·Frightened·Incapacitated·Invisible·Hide),
  집중 용어집, 몬스터 블록 p.256·283·290·319-320·324·340-341·347·349·359, 무기 p.91, 방어구 p.92,
  직업 p.36-38·47-48·61-62·77-78, 주문 p.121·132·138·139·146·157·159, 레벨업 고정 HP p.23.
위치 기반 규칙(엄폐, 사거리, 근접 중 원거리 불리, 기회 공격, 이동)은 쓰지 않는다 [05-7 4-2].
전투 진행 구조:
  start_combat(조우 id)  우선권을 굴려 순서를 정하고 s.combat에 전투 상태를 만든다.
  _run_until_hero()      몬스터 차례를 자동으로 진행하다가 주인공 차례가 되면 멈춘다.
  combat_choices()       주인공 차례의 선택지 (id는 모두 "cmb:"로 시작) → Engine.available_choices가 돌려준다.
  combat_choose(id)      행동 1개 처리. 행동·보너스 행동을 다 쓰면 자동으로 차례를 넘긴다.
  _end_combat(결과)       s.combat = None, 결과를 s.vars["combat:조우 id"]에 남기고 조우의 on_victory 등 효과를 실행.
데이터: 몬스터 수치 monsters.json · 조우 encounters.json · 무기/방어구/주문/직업 표 rules.json.
self.c / self.s / self.dice / _damage / _death_save 등은 Engine(engine.py)에 있는 것을 쓴다.
```

**들어 있는 것**
- **`_mode`** (함수) — 이점과 불리가 함께 있으면 상쇄 [SRD p.7].
- **`CombatMixin`** (클래스) — 전투 기능 묶음. 혼자서는 쓰지 않고 Engine이 상속해서 쓴다 (engine.py와 파일만 나눈 것).
- **`_add_dice`** (함수) — '4d6' + '1d6' × 2 → '6d6' (같은 주사위만).

In [ ]:
OUTCOME_KO = {"victory": "승리", "retreat": "물러남", "avoid": "전투 회피", "peace": "적대 해제", "defeat": "패배"}


DTYPE_KO = {"slashing": "베기", "piercing": "관통", "bludgeoning": "타격", "fire": "화염", "cold": "냉기",
            "radiant": "광휘", "necrotic": "사령", "acid": "산성", "force": "역장", "poison": "독", "psychic": "정신"}


TURN_UNDEAD_ROUNDS = 10        # 1분 [SRD p.37 Turn Undead]


FIEND_UNDEAD = ("fiend", "undead")   # Holy Water [SRD p.97]


def _mode(adv, dis):
    """이점과 불리가 함께 있으면 상쇄 [SRD p.7]."""
    if adv and dis:
        return "normal"
    return "advantage" if adv else "disadvantage" if dis else "normal"

In [ ]:
class CombatMixin:
    """전투 기능 묶음. 혼자서는 쓰지 않고 Engine이 상속해서 쓴다 (engine.py와 파일만 나눈 것)."""

    # ================================================================== 캐릭터 전투 수치
    def class_value(self, char, key, default=0):
        """rules.json class_levels에서 이 캐릭터의 직업·레벨 값을 읽는다 (예: 도적 sneak_dice)."""
        return self.c.rules["class_levels"].get(char.cls, {}).get(str(char.level), {}).get(key, default)

    def init_resources(self, char):
        """레벨 표의 사용 횟수로 채운다 (긴 휴식·레벨업)."""
        table = self.c.rules["class_levels"].get(char.cls, {}).get(str(char.level), {})
        char.resources = {k: (list(v) if isinstance(v, list) else v) for k, v in table.items() if k != "sneak_dice"}

    def armor_class(self, char):
        """[SRD p.92] 방어구 기본값 + 민첩(상한) + 방패 2, Defense +1 [SRD p.88], 점액 부식 −n [SRD p.290]."""
        arm = self.c.rules["armor"][char.armor]
        dex = char.mod("dex") if arm["dex_max"] is None else min(char.mod("dex"), arm["dex_max"])
        ac = arm["base"] + dex - (char.armor_penalty if char.armor != "none" else 0)
        if char.shield:
            ac += self.c.rules["shield_bonus"]
        if char.fighting_style == "defense" and char.armor != "none":
            ac += 1
        return ac

    def weapon_proficient(self, char, wid):
        """무기 숙련 여부. martial_finesse_light = 도적처럼 군용 무기 중 Finesse·Light 무기만 숙련."""
        w = self.c.rules["weapons"][wid]
        if w["category"] == "simple":
            return "simple" in char.weapon_profs
        if "martial" in char.weapon_profs:
            return True
        return "martial_finesse_light" in char.weapon_profs and ("finesse" in w["props"] or "light" in w["props"])

    def weapon_attack(self, char, wid):
        """(능력치, 명중 보너스). 근접=근력, 원거리=민첩, Finesse=둘 중 높은 것 [SRD p.15]."""
        w = self.c.rules["weapons"][wid]
        if "finesse" in w["props"]:
            ab = max(("str", "dex"), key=char.mod)
        elif w["kind"] == "ranged":
            ab = "dex"
        else:
            ab = "str"
        bonus = char.mod(ab) + (char.proficiency_bonus if self.weapon_proficient(char, wid) else 0)
        return ab, bonus - char.weapon_penalty.get(wid, 0)

    def spell_attack_bonus(self, char):
        return char.mod(char.spellcasting) + char.proficiency_bonus      # [SRD p.106]

    def spell_dc(self, char):
        return 8 + char.mod(char.spellcasting) + char.proficiency_bonus  # [SRD p.106]

    def _slot_level(self, char, level):
        """쓸 수 있는 가장 낮은 슬롯 레벨 (없으면 None). 상위 슬롯 사용은 자동 [보완]."""
        slots = char.resources.get("slots", [])
        for lv in range(level, len(slots) + 1):
            if slots[lv - 1] > 0:
                return lv
        return None

    # ================================================================== 레벨업 [SRD p.23]
    def _level_up(self, char, to_level):
        """to_level까지 한 단계씩 올린다 ({"level_up": n} 효과, 전투 시험장의 레벨 맞추기)."""
        events = []
        fixed = self.c.rules["fixed_hp_per_level"][char.cls]
        while char.level < to_level:
            char.level += 1
            gain = fixed + char.mod("con")   # 고정 HP 방식 [SRD p.23 Fixed Hit Points by Class]
            char.hp_max += gain
            char.hp += gain
            char.hd_left += 1
            self.init_resources(char)        # 새 레벨의 사용 횟수로 채움 [보완]
            events.append(Event("status", f"{char.name}가 {char.level}레벨이 되었다. 최대 HP +{gain} → {char.max_hp}"))
        return events

    # ================================================================== 전투 시작
    def start_combat(self, enc_id, on_end=None):
        """전투 시작. on_end = {"victory": [효과…], "retreat": [...]} — 장면 JSON의 on_victory 등을 Engine._apply가 넘겨준다.

        s.combat 구조: enc(조우 id), round, order(행동 순서 id 목록, 주인공은 "hero"), idx(지금 차례 위치),
        enemies(몬스터 id → hp·conditions·out 등), turn(주인공 남은 행동 수), xp, guided, dodging, on_end.
        """
        enc = self.c.encounters[enc_id]
        enemies, order = {}, []
        events = [Event("combat", f"⚔ 전투 시작: {enc['name']}", {"encounter": enc_id})]
        hero = self.hero
        surprised = bool(enc.get("surprise_unless_flag")) and enc["surprise_unless_flag"] not in self.s.flags
        adv = bool(enc.get("init_advantage_flag")) and enc["init_advantage_flag"] in self.s.flags
        d, rolls = self.dice.d20(_mode(adv, surprised))   # 기습당하면 우선권 불리 [SRD p.13]
        hero_init = d + hero.mod("dex")
        tag = " (기습당함: 불리)" if surprised and not adv else " (이점)" if adv and not surprised else ""
        events.append(Event("check", f"{hero.name}의 우선권: d20 {rolls if len(rolls) > 1 else d} {hero.mod('dex'):+d} = {hero_init}{tag}"))
        order.append(("hero", hero_init, 0))
        for gi, grp in enumerate(enc["monsters"]):
            md = self.c.monsters[grp["monster"]]
            gd, _ = self.dice.d20()                        # 같은 종류는 한 번만 굴린다 [SRD p.13]
            init = gd + md["init"]
            for i in range(grp["count"]):
                mid = f"{grp['monster']}_{i + 1}"
                name = md["name"] + (f" {i + 1}" if grp["count"] > 1 else "")
                enemies[mid] = {"id": mid, "kind": grp["monster"], "name": name, "hp": md["hp"], "hp_max": md["hp"],
                                "conditions": {}, "used": {}, "out": ""}
                order.append((mid, init, 1 + gi))
            events.append(Event("check", f"{md['name']}의 우선권: {init}"))
        order.sort(key=lambda x: (-x[1], x[2]))           # 동점이면 PC 먼저 [보완]
        self.s.combat = {"enc": enc_id, "round": 1, "order": [o[0] for o in order], "idx": 0, "enemies": enemies,
                         "turn": None, "xp": 0, "guided": None, "dodging": False, "on_end": on_end or {}}
        events.append(Event("combat", "순서: " + " → ".join(hero.name if o == "hero" else enemies[o]["name"]
                                                          for o in self.s.combat["order"])))
        if any("shriek" in self.c.monsters[m["kind"]]["traits"] for m in enemies.values()):
            self.s.vars["noise:shriekers"] = True          # 다가오면 비명 [SRD p.283, 원본: 아래쪽 전부가 안다]
            events.append(Event("say", "버섯이 귀를 찢는 비명을 지른다! 아래쪽 전부가 이 소리를 들었을 것이다."))
        return events + self._run_until_hero()

    def living_enemies(self):
        """아직 싸우는 적 (out이 ""인 것). out = "dead"(쓰러짐) / "fled"(달아남)."""
        return [m for m in self.s.combat["enemies"].values() if not m["out"]]

    def wound_text(self, m):
        """적 HP 숫자 대신 화면에 보여 줄 상태 말 (멀쩡함 / 다침 / 피투성이)."""
        if m["hp"] >= m["hp_max"]:
            return "멀쩡함"
        return "피투성이" if m["hp"] * 2 <= m["hp_max"] else "다침"     # Bloodied = 절반 이하 [SRD p.16]

    # ================================================================== 진행
    def _run_until_hero(self):
        """순서표를 따라 몬스터 차례를 자동 진행하고, 주인공이 행동할 수 있는 차례가 오면 멈춘다.
        전투가 끝나거나 게임 오버가 되면 바로 돌아간다. 5000번은 무한 반복을 막는 안전장치."""
        events = []
        for _ in range(5000):
            cb = self.s.combat
            if not cb or self.s.ended:
                return events
            who = cb["order"][cb["idx"]]
            if who == "hero":
                ev, ready = self._hero_turn_start()
                events += ev
                if ready:
                    return events
            else:
                events += self._monster_turn(who)
            if not self.s.combat or self.s.ended:
                return events
            events += self._check_end()
            if not self.s.combat:
                return events
            self._next_slot()
        raise RuntimeError("전투가 끝나지 않음")

    def _next_slot(self):
        cb = self.s.combat
        cb["idx"] += 1
        if cb["idx"] >= len(cb["order"]):
            cb["idx"] = 0
            cb["round"] += 1

    def _hero_turn_start(self):
        """주인공 차례 시작. (Event 목록, 선택지를 받을 준비가 됐는지)를 돌려준다.
        쓰러져 있으면 죽음 구원만 굴리고 False — 혼자 안정 상태가 되면 1인 패배로 게임 오버."""
        cb, hero = self.s.combat, self.hero
        cb["dodging"] = False                                # Dodge는 다음 턴 시작까지 [SRD 용어집]
        cb["turn"] = {"action": 1, "bonus": 1, "sneak": False, "surged": False}
        events = [Event("combat", f"— {cb['round']}라운드: {hero.name}의 차례 —", {"round": cb["round"]})]
        if not hero.down:
            return events, True
        # 쓰러짐: 자기 턴마다 죽음 구원 [SRD p.17-18]. 적은 쓰러진 캐릭터를 공격하지 않음 [보완 05-6 D2]
        if "dying" in hero.conditions:
            events += self._death_save(hero)
        if "dead" in hero.conditions:
            return events, False
        if not hero.down:
            events.append(Event("status", f"{hero.name}가 다시 일어선다!"))
            return events, True
        if "stable" in hero.conditions:                      # 혼자 쓰러진 채 안정 → 1인 패배 [보완 05-6 D4]
            events += self._end_combat("defeat")
            self.s.ended = "game_over"
            events.append(Event("end", f"전투에서 쓰러져 패배했다. 게임 오버. 체크포인트({self.checkpoint_label})에서 다시 시작할 수 있다."))
        return events, False

    def _end_hero_turn(self):
        cb = self.s.combat
        g = cb.get("guided")
        if g and g["round"] < cb["round"]:                 # Guiding Bolt: 다음 턴이 끝날 때까지 [SRD p.138]
            cb["guided"] = None
        self._next_slot()
        return self._run_until_hero()

    def _check_end(self):
        if self.s.combat and not self.living_enemies():
            return self._end_combat("victory")
        return []

    def _end_combat(self, outcome, text=None):
        """전투 종료. outcome: victory / retreat / avoid / peace / defeat (OUTCOME_KO).
        붙잡힘을 풀고, 결과를 s.vars["combat:조우 id"]에 기록하고, 패배가 아니면 시간 경과 + 조우의 on_결과 효과 실행."""
        cb = self.s.combat
        enc_id, rounds, hero = cb["enc"], cb["round"], self.hero
        self.s.combat = None
        hero.conditions.pop("grappled", None)
        if hero.conditions.get("restrained") == "grapple":
            hero.conditions.pop("restrained")
        events = [Event("say", text)] if text else []
        events.append(Event("combat", f"전투 종료: {OUTCOME_KO[outcome]} ({rounds}라운드)",
                            {"outcome": outcome, "encounter": enc_id, "rounds": rounds, "xp": cb["xp"]}))
        self.s.vars[f"combat:{enc_id}"] = outcome
        self.s.vars["xp"] = self.s.vars.get("xp", 0) + cb["xp"]
        if outcome != "defeat":
            events += self._pass_time(max(1, math.ceil(rounds * 6 / 60)))   # 1라운드 = 6초 [SRD p.13]
            events += self._apply(cb["on_end"].get(outcome, []), hero)
        return events

    # ================================================================== 선택지
    def combat_choices(self):
        """주인공 차례의 선택지. 남은 행동(action)·보너스 행동(bonus) 수에 따라 보이는 것이 달라진다.
        id 형식: cmb:attack:무기:적 / cmb:cast:주문:적|self / cmb:holy:적 / cmb:turn / cmb:escape / cmb:dodge
                 cmb:option:완화수단 / cmb:use:아이템 / cmb:wind / cmb:surge / cmb:end
        키: 행동은 숫자(목록 순번), 완화 수단 x숫자, 보너스 b숫자, Action Surge v, 차례 마침 z."""
        cb, hero = self.s.combat, self.hero
        t, out = cb["turn"], []
        foes = self.living_enemies()
        spells = self.c.rules["spells"]
        n = [0]

        def add(cid, label, key=None):
            if key is None:
                n[0] += 1
                key = str(n[0])
            out.append(ChoiceView(cid, label, [hero.id], hero.id, key=key))

        if t["action"] > 0:
            for wid in hero.weapons:
                w = self.c.rules["weapons"][wid]
                ab, bonus = self.weapon_attack(hero, wid)
                for m in foes:
                    add(f"cmb:attack:{wid}:{m['id']}", f"공격: {w['name']} → {m['name']} ({self.wound_text(m)}) [명중 {bonus:+d}, {w['damage']}{hero.mod(ab):+d}]")
            for sid in hero.cantrips + hero.spells:
                sp = spells[sid]
                if sp["time"] != "action" or (sp["level"] and self._slot_level(hero, sp["level"]) is None):
                    continue
                slot = f", 슬롯 {self._slot_level(hero, sp['level'])}" if sp["level"] else ""
                if sp["kind"] in ("attack", "save", "missile"):
                    for m in foes:
                        add(f"cmb:cast:{sid}:{m['id']}", f"주문: {sp['name']} → {m['name']} ({self.wound_text(m)}){slot}")
                elif sp["kind"] == "heal" and hero.hp < hero.max_hp:
                    add(f"cmb:cast:{sid}:self", f"주문: {sp['name']} (자신 치유{slot})")
                elif sp["kind"] == "ward" and hero.concentration != sid and self.s.has_item(sp["material"]):
                    add(f"cmb:cast:{sid}:self", f"주문: {sp['name']} (성수 1병 소모, 집중{slot})")
            if self.s.has_item("holy_water"):
                for m in foes:
                    add(f"cmb:holy:{m['id']}", f"성수 투척 → {m['name']} (남은 {self.s.inventory['holy_water']}병)")
            if ("turn_undead" in hero.features and hero.resources.get("channel_divinity", 0) > 0
                    and any(self.c.monsters[m["kind"]]["type"] == "undead" for m in foes)):
                add("cmb:turn", f"Turn Undead (Channel Divinity 남은 {hero.resources['channel_divinity']}회)")
            if "grappled" in hero.conditions:
                g = next(iter(hero.conditions["grappled"].values()))
                add("cmb:escape", f"붙잡힘에서 벗어난다 (근력(운동)/민첩(곡예) DC {g['dc']})")
            add("cmb:dodge", "회피 (다음 차례까지 공격받을 때 불리)")
            for i, opt in enumerate(self.c.encounters[cb["enc"]].get("options", []), 1):
                if opt["result"] == "retreat" and ("grappled" in hero.conditions or "restrained" in hero.conditions):
                    continue                                   # 속도 0이면 물러날 수 없다 [SRD 용어집 Grappled]
                if self.cond(opt.get("requires"), hero):
                    add(f"cmb:option:{opt['id']}", opt["label"], key=f"x{i}")
        if t["bonus"] > 0:
            for i, item_id in enumerate(self.c.use_order, 1):   # 물약은 보너스 행동 [SRD p.99]
                if self.s.has_item(item_id) and self._usable(hero, self.c.items[item_id]["use"]):
                    add(f"cmb:use:{item_id}", f"(보너스) {self.c.items[item_id]['use']['label']}", key=f"b{i}")
            if "second_wind" in hero.features and hero.resources.get("second_wind", 0) > 0 and hero.hp < hero.max_hp:
                add("cmb:wind", f"(보너스) Second Wind: 1d10+{hero.level} 회복 (남은 {hero.resources['second_wind']}회)", key="b8")
            for sid in hero.spells:
                sp = spells[sid]
                if sp["time"] == "bonus" and hero.hp < hero.max_hp and self._slot_level(hero, sp["level"]):
                    add(f"cmb:cast:{sid}:self", f"(보너스) 주문: {sp['name']} (자신 치유, 슬롯 {self._slot_level(hero, sp['level'])})", key="b9")
        if (t["action"] == 0 and not t["surged"] and "action_surge" in hero.features
                and hero.resources.get("action_surge", 0) > 0):
            add("cmb:surge", "Action Surge: 행동을 한 번 더 한다", key="v")
        add("cmb:end", "차례를 마친다", key="z")
        return out

    def combat_choose(self, cid):
        """전투 선택지 1개 처리 (Engine.choose가 "cmb:" id를 여기로 보낸다). 행동 수를 줄이고 결과를 적용한 뒤,
        적이 모두 쓰러졌는지 확인하고, 더 할 행동이 없으면 차례를 자동으로 넘긴다."""
        parts = cid.split(":")[1:]
        kind = parts[0]
        cb, hero = self.s.combat, self.hero
        t = cb["turn"]
        if kind == "end":
            return self._end_hero_turn()
        if kind == "option":
            opt = next(o for o in self.c.encounters[cb["enc"]]["options"] if o["id"] == parts[1])
            return self._end_combat(opt["result"], opt.get("text"))
        events = []
        if kind == "attack":
            t["action"] -= 1
            events += self._hero_weapon_attack(parts[1], cb["enemies"][parts[2]])
        elif kind == "cast":
            sp = self.c.rules["spells"][parts[1]]
            t["bonus" if sp["time"] == "bonus" else "action"] -= 1
            events += self._cast(parts[1], None if parts[2] == "self" else cb["enemies"][parts[2]])
        elif kind == "holy":
            t["action"] -= 1
            events += self._holy_water(cb["enemies"][parts[1]])
        elif kind == "turn":
            t["action"] -= 1
            events += self._turn_undead()
        elif kind == "escape":
            t["action"] -= 1
            events += self._escape_grapple()
        elif kind == "dodge":
            t["action"] -= 1
            cb["dodging"] = True
            events.append(Event("combat", f"{hero.name}가 몸을 낮추고 공격을 피할 준비를 한다."))
        elif kind == "use":
            t["bonus"] -= 1
            events += self._use_item(parts[1], hero)
        elif kind == "wind":                                   # [SRD p.47] 보너스 행동, 1d10 + 전사 레벨
            t["bonus"] -= 1
            hero.resources["second_wind"] -= 1
            r, _ = self.dice.roll("1d10")
            before, hero.hp = hero.hp, min(hero.max_hp, hero.hp + r + hero.level)
            events.append(Event("status", f"Second Wind: 1d10({r})+{hero.level} 회복. HP {before} → {hero.hp}/{hero.max_hp}"))
        elif kind == "surge":                                  # [SRD p.47] 행동 1회 추가 (Magic 행동 제외)
            hero.resources["action_surge"] -= 1
            t["action"] += 1
            t["surged"] = True
            events.append(Event("combat", f"{hero.name}가 한계를 넘어 한 번 더 움직인다! (Action Surge)"))
        if self.s.combat and not self.s.ended:
            events += self._check_end()
        if self.s.combat and not self.s.ended:
            if not [v for v in self.combat_choices() if v.id != "cmb:end"]:
                events += self._end_hero_turn()                # 더 할 것이 없으면 자동으로 차례 종료
        return events

    # ================================================================== PC의 공격
    def _roll_damage(self, expr, crit=False):
        """치명타: 피해 주사위를 두 번 굴림 (수정치는 한 번) [SRD p.15]."""
        m = DICE_EXPR.match(expr)
        count, sides = int(m.group(1) or 1), int(m.group(2))
        bonus = int(m.group(3).replace(" ", "")) if m.group(3) else 0
        rolls = [self.dice.die(sides) for _ in range(count * (2 if crit else 1))]
        return sum(rolls) + bonus, rolls

    def _hero_attack_flags(self, m):
        """주인공이 적 m을 공격할 때의 (이점, 불리). 둘 다 있으면 _mode가 상쇄한다."""
        hero, cb = self.hero, self.s.combat
        adv = bool(cb.get("guided") and cb["guided"]["mid"] == m["id"])
        adv = adv or "restrained" in m["conditions"]
        dis = "poisoned" in hero.conditions or "restrained" in hero.conditions      # [SRD 용어집]
        grapplers = hero.conditions.get("grappled", {})
        dis = dis or (bool(grapplers) and m["id"] not in grapplers)                  # Grappled [SRD 용어집]
        dis = dis or "hidden" in m["conditions"]                                     # 보이지 않는 대상 [SRD p.14]
        return adv, dis

    def _attack_roll(self, m, bonus, label):
        """PC → 몬스터 공격 굴림. (명중, 치명타, 모드, 이벤트)."""
        md = self.c.monsters[m["kind"]]
        adv, dis = self._hero_attack_flags(m)
        mode = _mode(adv, dis)
        if self.s.combat.get("guided") and self.s.combat["guided"]["mid"] == m["id"]:
            self.s.combat["guided"] = None                    # 다음 공격 1회에만 [SRD p.138]
        d, rolls = self.dice.d20(mode)
        total = d + bonus
        crit = d == 20
        hit = d != 1 and (crit or total >= md["ac"])          # 1은 빗나감, 20은 명중 [SRD p.15]
        tag = {"advantage": " (이점)", "disadvantage": " (불리)"}.get(mode, "")
        res = "치명타!" if crit else "명중" if hit else "빗나감"
        text = f"{self.hero.name}의 {label}: d20 {rolls if len(rolls) > 1 else d} {bonus:+d} = {total} vs AC {md['ac']} → {res}{tag}"
        return hit, crit, mode, Event("check", text, {"roll": d, "total": total, "ac": md["ac"], "hit": hit, "crit": crit})

    def _hero_weapon_attack(self, wid, m):
        """무기 공격: 공격 굴림 → 피해(+능력 수정치) → 조건이 맞으면 암습 → 몬스터에게 피해 → 점액이면 무기 부식."""
        hero, w = self.hero, self.c.rules["weapons"][wid]
        md = self.c.monsters[m["kind"]]
        ab, bonus = self.weapon_attack(hero, wid)
        hit, crit, mode, ev = self._attack_roll(m, bonus, f"{w['name']} 공격")
        events = [ev]
        if not hit:
            return events
        dmg, rolls = self._roll_damage(w["damage"], crit)
        dmg = max(0, dmg + hero.mod(ab))
        why = f"{w['damage']}({'+'.join(map(str, rolls))}){hero.mod(ab):+d}"
        t = self.s.combat["turn"]
        if ("sneak_attack" in hero.features and not t["sneak"] and mode == "advantage"
                and ("finesse" in w["props"] or w["kind"] == "ranged")):
            # Sneak Attack: 이점이 있고 Finesse·원거리 무기. 1인이라 '아군이 5ft 안' 조건은 쓰지 않음 [SRD p.61-62]
            nd = self.class_value(hero, "sneak_dice", 1)
            extra, er = self._roll_damage(f"{nd}d6", crit)
            dmg += extra
            t["sneak"] = True
            why += f" + 암습 {nd}d6({'+'.join(map(str, er))})"
        events += self._hurt_monster(m, dmg, w["type"], crit, why)
        if "corrosive_form" in md["traits"] and dmg > 0 and w["kind"] == "melee" and not w.get("thrown"):
            # 비마법 무기: 피해를 준 뒤 명중 −1 누적, −5면 파괴 [SRD p.290]
            hero.weapon_penalty[wid] = hero.weapon_penalty.get(wid, 0) + 1
            if hero.weapon_penalty[wid] >= 5:
                hero.weapons.remove(wid)
                events.append(Event("status", f"{w['name']}이(가) 점액에 녹아 부서졌다!"))
            else:
                events.append(Event("status", f"{w['name']}이(가) 부식되었다 (명중 −{hero.weapon_penalty[wid]})."))
        return events

    def _cast(self, sid, m):
        """주문 시전. rules.json spells의 kind로 처리 방식이 갈린다:
        attack(주문 공격 굴림) / save(대상 구원 실패 시 피해) / missile(자동 명중) / heal(자신 회복) / ward(보호 결계, 집중).
        1레벨 이상 주문은 쓸 수 있는 가장 낮은 슬롯을 쓰고, 높은 슬롯이면 upcast만큼 주사위가 늘어난다."""
        hero, sp = self.hero, self.c.rules["spells"][sid]
        events = []
        up = 0
        if sp["level"]:
            lv = self._slot_level(hero, sp["level"])
            hero.resources["slots"][lv - 1] -= 1
            up = lv - sp["level"]
            events.append(Event("status", f"{sp['name']} 시전 (주문 슬롯 {lv}레벨 사용, 남은 슬롯 {hero.resources['slots']})"))
        if sp["kind"] == "attack":
            hit, crit, _, ev = self._attack_roll(m, self.spell_attack_bonus(hero), f"{sp['name']} 주문 공격")
            events.append(ev)
            if hit:
                expr = sp["damage"] if not up else _add_dice(sp["damage"], sp["upcast"], up)
                dmg, rolls = self._roll_damage(expr, crit)
                events += self._hurt_monster(m, dmg, sp["type"], crit, f"{expr}({'+'.join(map(str, rolls))})")
                if sp.get("guide") and not m["out"]:
                    self.s.combat["guided"] = {"mid": m["id"], "round": self.s.combat["round"]}
                    events.append(Event("status", f"{m['name']}이(가) 빛에 휩싸였다. 다음 공격은 이점을 받는다."))
        elif sp["kind"] == "save":
            ok, ev = self._monster_save(m, sp["save"], self.spell_dc(hero))
            events.append(ev)
            if not ok:
                dmg, rolls = self._roll_damage(sp["damage"])
                events += self._hurt_monster(m, dmg, sp["type"], False, f"{sp['damage']}({'+'.join(map(str, rolls))})")
        elif sp["kind"] == "missile":                           # 자동 명중 [SRD p.146]
            darts = sp["darts"] + up
            total, all_rolls = 0, []
            for _ in range(darts):
                v, r = self._roll_damage(sp["damage"])
                total += v
                all_rolls += r
            events += self._hurt_monster(m, total, sp["type"], False, f"{darts}발 × {sp['damage']}({'+'.join(map(str, all_rolls))})")
        elif sp["kind"] == "heal":
            expr = sp["heal"] if not up else _add_dice(sp["heal"], sp["upcast"], up)
            v, r = self._roll_damage(expr)
            amount = v + hero.mod(hero.spellcasting)
            before, hero.hp = hero.hp, min(hero.max_hp, hero.hp + amount)
            events.append(Event("status", f"{sp['name']}: {expr}({'+'.join(map(str, r))}){hero.mod(hero.spellcasting):+d} 회복. HP {before} → {hero.hp}/{hero.max_hp}"))
        elif sp["kind"] == "ward":                              # [SRD p.157] 성수 소모, 집중 10분
            self.s.inventory[sp["material"]] -= 1
            self._end_concentration(hero)
            hero.concentration = sid
            hero.conditions[sid] = sp["minutes"]
            events.append(Event("status", f"{hero.name}가 보호의 결계를 두른다. 언데드 등은 {hero.name}를 공격할 때 불리 (집중)."))
        return events

    def _monster_save(self, m, ability, dc):
        """몬스터 구원 굴림 (수정치는 monsters.json saves). (성공 여부, Event)."""
        md = self.c.monsters[m["kind"]]
        d, _ = self.dice.d20()
        total = d + md["saves"][ability]
        ok = total >= dc
        return ok, Event("check", f"{m['name']}의 {ability.upper()} 구원: d20 {d} {md['saves'][ability]:+d} = {total} vs DC {dc} → {'성공' if ok else '실패'}")

    def _holy_water(self, m):
        """[SRD p.97] 공격 1회 대신. 민첩 구원 DC = 8 + 던진 사람 민첩 + 숙련, 실패 시 마귀·언데드에 2d8 광휘."""
        hero = self.hero
        self.s.inventory["holy_water"] -= 1
        dc = 8 + hero.mod("dex") + hero.proficiency_bonus
        events = [Event("item", f"{hero.name}가 성수를 {m['name']}에게 던진다. (남은 {self.s.inventory['holy_water']}병)")]
        ok, ev = self._monster_save(m, "dex", dc)
        events.append(ev)
        if not ok and self.c.monsters[m["kind"]]["type"] in FIEND_UNDEAD:
            dmg, rolls = self._roll_damage("2d8")
            events += self._hurt_monster(m, dmg, "radiant", False, f"2d8({'+'.join(map(str, rolls))})")
        elif not ok:
            events.append(Event("combat", "성수가 튀었지만 아무 효과가 없다."))
        return events

    def _turn_undead(self):
        """[SRD p.37] 30ft 안의 언데드: 지혜 구원 실패 → Frightened + Incapacitated 1분. 피해를 받으면 끝."""
        hero = self.hero
        hero.resources["channel_divinity"] -= 1
        events = [Event("combat", f"{hero.name}가 성표를 높이 든다! (Turn Undead)")]
        for m in self.living_enemies():
            md = self.c.monsters[m["kind"]]
            if md["type"] != "undead":
                continue
            ok, ev = self._monster_save(m, "wis", self.spell_dc(hero))
            events.append(ev)
            if ok:
                continue
            got = [c for c in ("frightened", "incapacitated") if c not in md["condition_immune"]]
            for c in got:
                m["conditions"][c] = TURN_UNDEAD_ROUNDS
            m["conditions"]["turned"] = TURN_UNDEAD_ROUNDS
            names = {"frightened": "겁에 질림", "incapacitated": "행동 불능"}
            events.append(Event("status", f"{m['name']}: {', '.join(names[c] for c in got) or '효과 없음 (면역)'}"))
        return events

    def _escape_grapple(self):
        """[SRD p.182] 행동으로 근력(운동) 또는 민첩(곡예) 판정 vs 탈출 DC."""
        hero = self.hero
        mid, g = next(iter(hero.conditions["grappled"].items()))
        ok, ev = self.roll_check(hero, {"options": [{"ability": "str", "skill": "athletics"},
                                                    {"ability": "dex", "skill": "acrobatics"}], "dc": g["dc"]})
        events = [ev]
        if ok:
            self._release(mid)
            events.append(Event("status", f"{hero.name}가 {self.s.combat['enemies'][mid]['name']}에게서 빠져나왔다."))
        return events

    def _release(self, mid):
        """몬스터 mid가 주인공을 붙잡고 있었다면 풀어 준다 (탈출 성공, 그 몬스터가 쓰러짐·달아남).
        hero.conditions["grappled"] = {몬스터 id: {"dc", "restrained"}} — 여러 몬스터가 동시에 붙잡을 수 있는 구조."""
        hero = self.hero
        grapplers = hero.conditions.get("grappled")
        if not grapplers or mid not in grapplers:
            return
        grapplers.pop(mid)
        if not grapplers:
            hero.conditions.pop("grappled")
        if hero.conditions.get("restrained") == "grapple" and not any(g.get("restrained") for g in grapplers.values()):
            hero.conditions.pop("restrained")

    def _hurt_monster(self, m, dmg, dtype, crit, why):
        """몬스터가 피해를 받는 유일한 통로. 면역 → 0, 저항 → 절반, 취약 → 2배를 적용하고,
        0 HP면 Undead Fortitude(좀비) 구원을 먼저 본 뒤 쓰러뜨린다. 문어는 피해를 받으면 먹물을 뿜고 달아난다."""
        md = self.c.monsters[m["kind"]]
        note = ""
        if dtype in md["immune"]:
            dmg, note = 0, " · 면역"
        elif dtype in md["resist"]:
            dmg, note = dmg // 2, " · 저항"
        elif dtype in md["vulnerable"]:
            dmg, note = dmg * 2, " · 취약"
        m["hp"] = max(0, m["hp"] - dmg)
        events = [Event("damage", f"{m['name']}에게 {DTYPE_KO.get(dtype, dtype)} 피해 {dmg} ({why}{note}) → {self.wound_text(m) if m['hp'] else '쓰러짐'}",
                        {"target": m["id"], "amount": dmg, "hp": m["hp"]})]
        if dmg > 0 and "turned" in m["conditions"]:            # Turn Undead는 피해를 받으면 끝 [SRD p.37]
            for c in ("turned", "frightened", "incapacitated"):
                m["conditions"].pop(c, None)
        if m["hp"] == 0:
            if "undead_fortitude" in md["traits"] and dtype != "radiant" and not crit and dmg > 0:
                ok, ev = self._monster_save(m, "con", 5 + dmg)   # DC 5 + 받은 피해 [SRD p.340]
                events.append(ev)
                if ok:
                    m["hp"] = 1
                    events.append(Event("status", f"{m['name']}이(가) 쓰러졌다가 다시 몸을 일으킨다! (Undead Fortitude)"))
                    return events
            m["out"] = "dead"
            self.s.combat["xp"] += md["xp"]
            self._release(m["id"])
            events.append(Event("status", f"{m['name']}이(가) 쓰러졌다."))
        elif (dmg > 0 and "ink_cloud" in md["traits"] and not m["used"].get("ink")
              and self.c.encounters[self.s.combat["enc"]].get("underwater")):
            # Ink Cloud (1/Day): 물속에서 피해를 받으면 먹물을 뿜고 헤엄쳐 감 [SRD p.349, 원본 scn-v8-octopus]
            m["used"]["ink"] = True
            m["out"] = "fled"
            self._release(m["id"])
            events.append(Event("status", f"{m['name']}이(가) 먹물 구름을 뿜고 물속 깊이 숨어 버렸다."))
        return events

    # ================================================================== 몬스터 차례
    def _monster_turn(self, mid):
        """몬스터 1마리의 차례. 남은 라운드가 있는 상태를 1 줄이고, 행동 불능이 아니면 첫 번째 공격을 multiattack번 한다.
        몬스터 AI는 단순하다: 대상은 항상 주인공, 쓰러진 주인공은 공격하지 않는다."""
        cb = self.s.combat
        m = cb["enemies"][mid]
        if m["out"]:
            return []
        md = self.c.monsters[m["kind"]]
        for c, left in list(m["conditions"].items()):
            if isinstance(left, int):
                m["conditions"][c] = left - 1
                if m["conditions"][c] <= 0:
                    del m["conditions"][c]
        if "incapacitated" in m["conditions"]:
            return [Event("combat", f"{m['name']}은(는) 겁에 질려 움직이지 못한다.")]
        if not md["attacks"]:
            return []
        hero = self.hero
        if hero.down:                                            # 쓰러진 캐릭터는 공격하지 않음 [보완 05-6 D2]
            return []
        events = []
        for _ in range(md["multiattack"]):                       # Multiattack [SRD 몬스터 블록]
            if hero.down or self.s.ended:
                break
            events += self._monster_attack(m, md["attacks"][0])
        if "shadow_stealth" in md["traits"] and self.c.encounters[cb["enc"]].get("dim_light"):
            # Shadow Stealth: 어둑한 빛·어둠에서 보너스 행동으로 Hide (민첩(은신) DC 15) [SRD p.320, 용어집 Hide]
            d, _ = self.dice.d20()
            if d + md.get("stealth", 0) >= 15:
                m["conditions"]["hidden"] = None
                events.append(Event("combat", f"{m['name']}이(가) 어둠 속으로 스며들어 보이지 않는다."))
        return events

    def _monster_attack(self, m, atk):
        """몬스터 공격 1회: 명중 굴림 vs 주인공 AC → 피해(Engine._damage) → 공격의 추가 효과
        (최대 HP 흡수, 근력 흡수, 붙잡기, 방어구 부식). 추가 효과는 monsters.json attacks의 키로 켜진다."""
        cb, hero = self.s.combat, self.hero
        md = self.c.monsters[m["kind"]]
        adv = "hidden" in m["conditions"] or "restrained" in hero.conditions
        dis = cb["dodging"]
        ward = "protection_from_evil_and_good"
        if ward in hero.conditions and md["type"] in self.c.rules["spells"][ward]["vs_types"]:
            dis = True                                           # [SRD p.157]
        m["conditions"].pop("hidden", None)                      # 공격하면 숨은 상태가 끝남 [SRD 용어집 Hide]
        mode = _mode(adv, dis)
        ac = self.armor_class(hero)
        d, rolls = self.dice.d20(mode)
        total = d + atk["to_hit"]
        crit = d == 20
        hit = d != 1 and (crit or total >= ac)
        tag = {"advantage": " (이점)", "disadvantage": " (불리)"}.get(mode, "")
        events = [Event("check", f"{m['name']}의 {atk['name']}: d20 {rolls if len(rolls) > 1 else d} {atk['to_hit']:+d} = {total} vs AC {ac} → "
                                 f"{'치명타!' if crit else '명중' if hit else '빗나감'}{tag}", {"hit": hit, "crit": crit})]
        if not hit:
            return events
        expr = atk["damage"]
        if atk.get("bloodied_damage") and m["hp"] * 2 <= m["hp_max"]:
            expr = atk["bloodied_damage"]                        # 쥐 떼: 피투성이면 1d4 [SRD p.359]
        dmg, r = self._roll_damage(expr, crit)
        why = f"{m['name']}의 {atk['name']} {expr}({'+'.join(map(str, r))}) {DTYPE_KO.get(atk['type'], atk['type'])}"
        if self.resists(hero, atk["type"]):
            dmg //= 2
            why += " · 저항"
        before_down = hero.down
        events += self._damage(hero, dmg, why, critical=crit)
        if "dead" in hero.conditions:
            return events
        if atk.get("max_hp_drain") and dmg > 0:                  # Life Drain [SRD p.324]
            hero.hp_max_loss += dmg
            hero.hp = min(hero.hp, hero.max_hp)
            events.append(Event("status", f"{hero.name}의 최대 HP가 {dmg} 줄었다 → {hero.max_hp}"))
        if atk.get("str_drain"):                                 # Draining Swipe [SRD p.320]
            n, _ = self.dice.roll(atk["str_drain"])
            hero.ability_loss["str"] = hero.ability_loss.get("str", 0) + n
            events.append(Event("status", f"{hero.name}의 근력이 {n} 줄었다 → {hero.score('str')}"))
            if hero.score("str") <= 0:
                return events + self._die(hero, "근력 0 — 원본: 그 자리에서 새 그림자가 된다")
        if hero.down or before_down:
            return events
        if atk.get("grapple"):                                   # [SRD 블록: 중형 이하 대상, 탈출 DC]
            g = hero.conditions.setdefault("grappled", {})
            g[m["id"]] = {"dc": atk["grapple"]["dc"], "restrained": bool(atk["grapple"].get("restrained"))}
            text = f"{hero.name}가 {m['name']}에게 붙잡혔다 (Grappled, 탈출 DC {atk['grapple']['dc']})"
            if atk["grapple"].get("restrained"):
                hero.conditions.setdefault("restrained", "grapple")
                text += " · Restrained"
            events.append(Event("status", text + "."))
        if atk.get("corrode_armor") and hero.armor != "none":    # 비마법 방어구 AC −1, 10이 되면 파괴 [SRD p.290]
            hero.armor_penalty += 1
            arm = self.c.rules["armor"][hero.armor]
            dex = hero.mod("dex") if arm["dex_max"] is None else min(hero.mod("dex"), arm["dex_max"])
            if arm["base"] + dex - hero.armor_penalty <= 10:
                events.append(Event("status", f"{arm['name']}이(가) 녹아 부서졌다!"))
                hero.armor, hero.armor_penalty = "none", 0
            else:
                events.append(Event("status", f"{arm['name']}이(가) 부식되었다 (AC −{hero.armor_penalty} → {self.armor_class(hero)})."))
        return events

    # ================================================================== 집중 [SRD 용어집 Concentration]
    def _end_concentration(self, char):
        """집중 중인 주문을 끝낸다 (주문 효과는 같은 이름의 condition으로 저장되어 있어 그것을 지운다)."""
        if char.concentration:
            char.conditions.pop(char.concentration, None)
            char.concentration = ""

    def _concentration_check(self, char, amount):
        """피해를 받았을 때 집중 유지 건강 구원 (DC = 피해의 절반, 10~30). Engine._damage가 부른다."""
        if not char.concentration or amount <= 0:
            return []
        name = self.c.rules["spells"][char.concentration]["name"]
        if char.down:
            self._end_concentration(char)
            return [Event("status", f"{name}의 집중이 끊겼다.")]
        dc = min(30, max(10, amount // 2))
        ok, ev = self.roll_save(char, "con", dc)
        if ok:
            return [ev]
        self._end_concentration(char)
        return [ev, Event("status", f"{name}의 집중이 끊겼다.")]

In [ ]:
def _add_dice(base, extra, times):
    """'4d6' + '1d6' × 2 → '6d6' (같은 주사위만)."""
    b, e = re.match(r"(\d+)d(\d+)", base), re.match(r"(\d+)d(\d+)", extra)
    return f"{int(b.group(1)) + int(e.group(1)) * times}d{b.group(2)}" + base[b.end():]

## [6] 게임 엔진 (Engine) — 선택지·판정·효과·카드 공개·이동
원본 파일: `game/engine.py`

```text
게임 엔진: 선택지 → 판정 → 효과(상태·flag·카드) → 다음 선택지.
규칙 근거 표기:
  [원본]  시나리오 카드        [SRD p.x] 실제로 읽어 확인한 SRD 쪽
  [보완]  개발 중 추가 규칙 (analysis/05_시스템설계/05-2a, 05-6)
모든 행동은 choose(choice_id, actor_id) 한 곳으로 들어온다. 나중에 자연어 입력을 붙일 때도
"자연어 → 현재 선택지 ID" 해석만 앞에 두면 이 경로를 그대로 쓴다.
한 번의 행동이 처리되는 순서 (화면·CLI·서버 모두 같다):
  1. available_choices()  지금 고를 수 있는 선택지(ChoiceView) 목록을 만든다.
  2. choose(id)           선택지 id의 앞부분(move:, buy:, cmb: …)을 보고 처리 함수로 보낸다.
                          장면 데이터의 선택지(접두어 없음)는 _run_choice → 판정 → _apply(효과 목록).
  3. 결과는 Event 목록으로 돌려준다. 상태(self.s)는 이미 바뀐 뒤이다.
이 파일이 쓰는 다른 파일:
  content.py (읽기 전용 데이터) · model.py (GameState·Character) · dice.py (주사위)
  events.py (Event·ChoiceView) · combat.py (CombatMixin: 전투 함수는 그쪽에 있고 Engine이 상속)
```

**들어 있는 것**
- **`Engine`** (클래스) — 게임 한 판. 규칙 판정과 상태 변경은 모두 이 객체가 한다.
- **`fmt_money`** (함수) — cp 단위 정수를 화면용 글자로. 예: 1250 → '12 gp 5 sp'. 돈은 항상 cp 정수로 저장한다.

In [ ]:
SKILL_KO = {
    "athletics": "운동", "acrobatics": "곡예", "sleight_of_hand": "손재주", "stealth": "은신",
    "arcana": "비전", "history": "역사", "investigation": "수사", "nature": "자연", "religion": "종교",
    "animal_handling": "동물 조련", "insight": "통찰", "medicine": "의학", "perception": "감지",
    "survival": "생존", "deception": "기만", "intimidation": "위협", "performance": "공연",
    "persuasion": "설득",
}


TOOL_KO = {"thieves_tools": "도둑 도구"}


LONG_REST_MIN = 8 * 60          # [SRD Rules Glossary: Long Rest] 최소 8시간


LONG_REST_GAP_MIN = 16 * 60     # [SRD Rules Glossary: Long Rest] 끝난 뒤 16시간 대기


SHORT_REST_MIN = 60             # [SRD Rules Glossary: Short Rest] 1시간


# 종족 특성 → 피해 저항 / 상태를 피하는 구원에 이점. Dwarven Resilience [SRD p.84]
TRAIT_RESIST = {"dwarven_resilience": {"poison"}}


TRAIT_SAVE_ADV = {"dwarven_resilience": {"poisoned"}}


# 지속 효과(조건) → 그 상태를 피하는 구원에 이점. Antitoxin [SRD p.94]
EFFECT_SAVE_ADV = {"antitoxin": {"poisoned"}}


EXPIRE_TEXT = {"poisoned": "{name}의 중독이 풀렸다.", "antitoxin": "{name}가 마신 해독제의 효과가 끝났다."}


DAMAGE_KO = {"poison": "독", "bludgeoning": "타격"}


SENTENCE_SPLIT = re.compile(r'(?<=[.!?。])\s+|(?<=[.!?]")\s+')


PERIAPT_WOUND = "periapt_wound_closure"   # 장면 2 전사 아이템 [원본]. 동조 절차는 생략(얻으면 바로 착용) [보완 W7]


MOVE_KEYS = "abcde"            # 이동 선택지의 고정 키 (s/k/q/h/r/w와 겹치지 않음)

In [ ]:
class Engine(CombatMixin):
    """게임 한 판. 규칙 판정과 상태 변경은 모두 이 객체가 한다.

    self.c     Content    : 장면·카드·아이템 등 읽기 전용 데이터 (게임 중 바뀌지 않음)
    self.s     GameState  : 이번 판의 모든 바뀌는 값 (위치, flag, HP, 소지품 …). 체크포인트는 이것의 복사본
    self.dice  Dice       : 주사위 (테스트에서는 FixedDice로 값을 정해 넣는다)
    self.opening          : 게임 시작 때 첫 방에 들어가며 나온 Event 목록 (화면이 처음에 보여 준다)
    """

    def __init__(self, content=None, dice=None, seed=None, char_id=None):
        if char_id is None:
            raise ValueError("플레이할 캐릭터를 골라야 합니다 (char_id). 1인 1캐릭터 [05-6 ①]")
        self.c = content or Content()
        self.dice = dice or Dice(seed)
        start = self.c.start_room
        self.s = GameState(party=self.c.new_party(char_id), room=start, scene=self.c.start_scene)
        self.s.vars["last_long_rest_end"] = None
        self.s.stock = {k: v.get("stock") for k, v in self.c.items.items() if "price_cp" in v}
        for m in self.s.party:
            self.init_resources(m)    # 주문 슬롯·Second Wind 등 (W6)
        self.checkpoint = None
        self.checkpoint_label = ""
        self.opening = self._enter(start)
        self._save_checkpoint("게임 시작")

    @property
    def hero(self):
        """플레이어 캐릭터. 1인 1캐릭터라 파티는 항상 한 명이다 (party 목록 구조는 핫시트용으로 남겨 둠)."""
        return self.s.party[0]

    # ------------------------------------------------------------------ 카드 열람/표시 (05-2a 3장)
    def eligible(self, card_id, char=None):
        """열람 가능: 기획서의 2축 필터(장면·대상) + flag + 방. char가 주어지면 그 캐릭터 기준."""
        card = self.c.cards[card_id]
        s = self.s
        if s.scene < card["reveal_after"]:
            return False
        if card["req_flag"] and card["req_flag"] not in s.flags:
            return False
        if card["room"] and card["room"] not in s.visited:
            return False
        if card["owner"] != "all":
            members = [char] if char else s.party
            if not any(m.cls == card["owner"] for m in members):
                return False
        return True

    def _view_text(self, card_id):
        """플레이어용 문장. card_views(보완)가 있으면 조건부 반응 문장을 빼고, 조건이 맞는 문장만 붙인다.
        문장은 원본 player_text의 문장을 그대로 쓴다(새 문장을 만들지 않음)."""
        card = self.c.cards[card_id]
        view = self.c.card_views.get(card_id)
        if not view:
            return card["player_text"], []
        sents = [x for x in SENTENCE_SPLIT.split(card["player_text"]) if x.strip()]
        hide = view.get("drop", []) + [p["match"] for p in view.get("parts", [])]
        kept = [x for x in sents if not any(h in x for h in hide)]
        later = []
        for i, part in enumerate(view.get("parts", [])):
            text = " ".join(x for x in sents if part["match"] in x)
            if self.cond(part.get("when")):
                kept.append(text)
            else:
                later.append((f"{card_id}#{i}", text, part.get("when")))
        return " ".join(kept), later

    def _show(self, card_id, actor=None):
        """카드를 플레이어에게 보여 준다 ({"show": 카드 id} 효과, 방 진입 묘사).

        - 이미 보여 준 카드, player_text가 빈 GM 전용 카드는 무시한다.
        - 아직 열람 조건(장면·flag)이 안 되면 deferred에 적어 두었다가, 이 방에서 '다시 둘러보기'로 보여 준다.
        - 보여 준 카드는 s.shown[카드 id] = [알게 된 캐릭터 id]로 기록한다. 화면의 단서 탭과
          선택지 조건 {"shown": …}이 이 기록을 쓴다.
        """
        card = self.c.cards[card_id]
        if card_id in self.s.shown or not card["player_text"]:
            return []
        if not self.eligible(card_id):
            # 공개 시점·flag가 아직이면 나중에 이 방에서 다시 볼 수 있게 기록 (지연 공개)
            if card["owner"] == "all" or any(m.cls == card["owner"] for m in self.s.party):
                self.s.deferred[card_id] = {"room": self.s.room, "card": card_id}
            return []
        self.s.deferred.pop(card_id, None)
        if card["owner"] == "all":
            knowers = [m.id for m in self.s.party]
            who = ""
        else:  # 역할 전용 카드: 그 역할 캐릭터만 앎 (플레이어 축)
            knowers = [m.id for m in self.s.party if m.cls == card["owner"]]
            who = "·".join(self.s.member(k).name for k in knowers)
        text, later = self._view_text(card_id)
        for key, part_text, when in later:
            self.s.deferred[key] = {"room": self.s.room, "part": part_text, "when": when, "card": card_id}
        self.s.shown[card_id] = knowers
        head = f"[{card['label']}]" + (f" ({who}만 알아챘다)" if who else "")
        return [Event("card", f"{head} {text}", {"id": card_id, "known_by": knowers})]

    def _ready_deferred(self):
        """지연 공개 목록 중 '지금 이 방에서' 보여 줄 수 있게 된 것의 key 목록 ('r' 선택지를 띄울지 판단)."""
        out = []
        for key, d in self.s.deferred.items():
            if d["room"] != self.s.room:
                continue
            if "part" in d:
                if d["card"] in self.s.shown and self.cond(d.get("when")):
                    out.append(key)
            elif self.eligible(d["card"]):
                out.append(key)
        return out

    def _revisit(self):
        """'다시 둘러보며 새 소식을 듣는다'(revisit:방) — 준비된 지연 카드·문장을 한꺼번에 보여 준다."""
        events = []
        for key in self._ready_deferred():
            d = self.s.deferred.pop(key)
            if "part" in d:
                card = self.c.cards[d["card"]]
                events.append(Event("card", f"[{card['label']}] {d['part']}", {"id": d["card"], "part": key}))
            else:
                events += self._show(d["card"])
        return events

    def player_known_cards(self, char_id=None):
        """플레이어에게 이미 보여 준 카드 id 목록 (char_id를 주면 그 캐릭터가 아는 것만). 서버 단서 탭·CLI 'k'가 쓴다."""
        return [cid for cid, ks in self.s.shown.items() if char_id is None or char_id in ks]

    # ------------------------------------------------------------------ 조건식 (05-2 4장)
    def cond(self, c, actor=None):
        """장면 데이터의 조건식을 참/거짓으로 계산한다. None(조건 없음)은 항상 참.

        조건식은 키 하나짜리 dict이며 all/any/not으로 묶을 수 있다. 예:
          {"all": [{"item": "rope"}, {"not": {"var": "rope_anchored"}}]}  → 밧줄이 있고, 아직 고정하지 않았다
        쓰는 곳: 선택지 requires, 출구 exit_requires, 효과 {"if": …}, 방 진입 on_enter, card_views의 when.
        새 조건 종류를 추가하면 데이터에서 바로 쓸 수 있다 (모르는 키는 ValueError로 바로 드러난다).
        """
        if c is None:
            return True
        if "all" in c:
            return all(self.cond(x, actor) for x in c["all"])
        if "any" in c:
            return any(self.cond(x, actor) for x in c["any"])
        if "not" in c:
            return not self.cond(c["not"], actor)
        if "flag" in c:
            return c["flag"] in self.s.flags
        if "var" in c:
            return bool(self.s.vars.get(c["var"]))
        if "item" in c:
            return self.s.has_item(c["item"])
        if "shown" in c:
            return c["shown"] in self.s.shown
        if "scene_at_least" in c:
            return self.s.scene >= c["scene_at_least"]
        if "darkvision" in c:  # 행동하는 캐릭터(없으면 주인공)에게 암시야가 있는지 (W7)
            return (actor or self.hero).darkvision > 0
        if "room_loaded" in c:  # 그 방의 장면 파일이 불러와졌는지 (아직 없는 장면으로 가는 임시 선택지용)
            return c["room_loaded"] in self.c.rooms
        if "trap_active" in c:
            return self.trap_active(c["trap_active"])
        if "condition" in c:  # 행동하는 캐릭터(없으면 주인공)의 상태
            return c["condition"] in (actor or self.hero).conditions
        raise ValueError(f"알 수 없는 조건: {c}")

    # ------------------------------------------------------------------ 판정
    def check_mod(self, char, chk):
        """(수정치, 설명). 숙련이면 숙련 보너스 추가 [SRD p.8]."""
        ab = chk["ability"]
        mod = char.mod(ab)
        name = ABILITY_KO[ab]
        prof = False
        if "skill" in chk:
            prof = chk["skill"] in char.skill_profs
            name += f"({SKILL_KO[chk['skill']]})"
        elif "tool" in chk:
            prof = chk["tool"] in char.tool_profs
            name += f"·{TOOL_KO.get(chk['tool'], chk['tool'])}"
        if prof:
            mod += char.proficiency_bonus
        return mod, name

    def best_option(self, char, chk):
        """판정에 여러 방법이 있으면(options, 예: 운동 또는 곡예) 이 캐릭터에게 수정치가 가장 높은 것을 고른다."""
        if "options" in chk:
            opts = [dict(o, dc=chk["dc"]) for o in chk["options"]]
            return max(opts, key=lambda o: self.check_mod(char, o)[0])
        return chk

    def roll_check(self, char, chk):
        """능력 판정: d20 + 수정치 ≥ DC 이면 성공. (성공 여부, 화면용 check Event)를 돌려준다.
        chk 예: {"ability": "str", "skill": "athletics", "dc": 15}. 상태는 바꾸지 않는다 (결과 적용은 호출한 쪽)."""
        chk = self.best_option(char, chk)
        mod, name = self.check_mod(char, chk)
        # 중독: 능력 판정에 불리 [SRD Rules Glossary: Poisoned]
        mode = "disadvantage" if "poisoned" in char.conditions else "normal"
        d, rolls = self.dice.d20(mode)
        total = d + mod
        ok = total >= chk["dc"]
        tag = " (중독: 불리)" if mode == "disadvantage" else ""
        text = (f"{char.name}의 {name} 판정: d20 {rolls if len(rolls) > 1 else d} "
                f"{mod:+d} = {total} vs DC {chk['dc']} → {'성공' if ok else '실패'}{tag}")
        return ok, Event("check", text, {"char": char.id, "roll": d, "rolls": rolls, "mod": mod,
                                          "total": total, "dc": chk["dc"], "success": ok})

    def save_mode(self, char, ability, avoids=None, advantage=False):
        """구원 굴림의 이점/불리. 둘 다 있으면 상쇄 [SRD p.7]."""
        adv = advantage
        if avoids:
            adv = adv or any(avoids in TRAIT_SAVE_ADV.get(t, ()) for t in char.traits)
            adv = adv or any(avoids in EFFECT_SAVE_ADV.get(cn, ()) for cn in char.conditions)
        dis = ability == "dex" and "restrained" in char.conditions  # Restrained: 민첩 구원에 불리 [SRD Rules Glossary]
        if adv and dis:
            return "normal"
        return "advantage" if adv else "disadvantage" if dis else "normal"

    def roll_save(self, char, ability, dc, advantage=False, avoids=None):
        """구원 굴림 (함정·몬스터 효과·집중 유지). avoids = 이 구원으로 피하려는 상태 이름 (드워프·해독제 이점 판단용)."""
        mod = char.mod(ability) + (char.proficiency_bonus if ability in char.save_profs else 0)
        mode = self.save_mode(char, ability, avoids, advantage)
        d, rolls = self.dice.d20(mode)
        total = d + mod
        ok = total >= dc
        tag = {"advantage": " (이점)", "disadvantage": " (불리)"}.get(mode, "")
        text = (f"{char.name}의 {ABILITY_KO[ability]} 구원 굴림: d20 {rolls if len(rolls) > 1 else d} "
                f"{mod:+d} = {total} vs DC {dc} → {'성공' if ok else '실패'}{tag}")
        return ok, Event("check", text, {"char": char.id, "total": total, "dc": dc, "success": ok, "save": ability})

    # ------------------------------------------------------------------ 선택지 목록
    def _actor_pool(self, ch):
        """이 선택지를 할 수 있는 캐릭터 목록. 비어 있으면 선택지가 화면에 나오지 않는다.
        actor 값: any(기본) / party(누가 하든 상관없는 일행 행동) / cls:직업 / not_cls:직업 / tool:도구."""
        spec = ch.get("actor", "any")
        if spec == "party":
            return []
        pool = [m for m in self.s.party if m.can_act()]
        if spec.startswith("cls:"):
            pool = [m for m in pool if m.cls == spec[4:]]
        elif spec.startswith("not_cls:"):
            pool = [m for m in pool if m.cls != spec[8:]]
        elif spec.startswith("tool:"):
            tool = spec[5:]
            pool = [m for m in pool if tool in m.tools or self.s.has_item(tool)]
        # M3: 같은 캐릭터는 실패한 판정을 다시 할 수 없다 [보완]
        return [m for m in pool if (ch["id"], m.id) not in self.s.failed]

    def available_choices(self):
        """지금 고를 수 있는 선택지(ChoiceView) 목록. 화면·CLI·서버가 매 턴 이 함수 결과를 그대로 보여 준다.

        우선순위 (위에서 걸리면 그 목록만 돌려준다):
          게임 끝 → 없음 / 결정 대기(pending: 영감 재굴림·Hit Point Dice) → 전투 중(combat_choices)
          → 쓰러진 동료 안정시키기 → 평상시 목록
        평상시 목록의 순서와 키:
          방의 선택지(숫자 키, 데이터 순번이라 끝난 것이 빠져도 번호가 당겨지지 않음) → 다시 둘러보기(r)
          → 깨어나기(w) → 소모품(u숫자) → 짧은 휴식(t) → 상점(p숫자) → 이동(a~e)
        """
        if self.s.ended:
            return []
        p = self.s.pending
        if p and p["type"] == "inspire":  # 실패한 판정을 영웅적 영감으로 다시 굴릴지 먼저 정한다
            return [ChoiceView("inspire:yes", "영웅적 영감을 써서 다시 굴린다", [], key="i"),
                    ChoiceView("inspire:no", "결과를 받아들인다", [], key="n")]
        if p and p["type"] == "hit_dice":  # 짧은 휴식: Hit Point Die를 하나씩 굴리고 더 쓸지 정한다 [SRD Short Rest]
            h = self.hero
            out = []
            if h.hd_left > 0 and h.hp < h.max_hp:
                out.append(ChoiceView("hd:spend", f"Hit Point Die 1개를 쓴다 (d{h.hit_die}{h.mod('con'):+d}, 남은 {h.hd_left}개)", [], key="t"))
            out.append(ChoiceView("hd:done", "짧은 휴식을 끝낸다", [], key="n"))
            return out
        if self.s.combat:
            return self.combat_choices()
        dying = [m for m in self.s.party if m.down and m.conditions.get("dying") is not None]
        if dying:  # 동료가 있으면 안정시키기가 먼저 [SRD p.18] (1인에서는 자동 처리되어 여기에 오지 않음)
            return self._stabilize_choices(dying[0])
        out = []
        for idx, ch in enumerate(self.c.choices_in_room(self.s.room), 1):  # 키 = 그 방 선택지 목록에서의 고정 순번
            if ch["id"] in self.s.done or not self.cond(ch.get("requires")):
                continue
            pool = self._actor_pool(ch)
            if ch.get("actor", "any") != "party" and not pool:
                continue
            rec, chk_text = "", ""
            if pool:
                if "check" in ch:
                    best = max(pool, key=lambda m: self.check_mod(m, self.best_option(m, ch["check"]))[0])
                    rec = best.id
                    mod, name = self.check_mod(best, self.best_option(best, ch["check"]))
                    chk_text = f"{name} DC {ch['check']['dc']}, {best.name} {mod:+d}"
                else:
                    rec = pool[0].id
            out.append(ChoiceView(ch["id"], ch["label"], [m.id for m in pool], rec, chk_text, key=str(idx)))
        if self._ready_deferred():
            out.append(ChoiceView(f"revisit:{self.s.room}", "다시 둘러보며 새 소식을 듣는다", [], key="r"))
        for stable in [m for m in self.s.party if m.down and "stable" in m.conditions]:
            out.append(ChoiceView(f"wake:{stable.id}", f"안정된 {stable.name}가 깨어날 때까지 기다린다 (1d4시간)", [], key="w"))
        hero = self.hero
        for i, item_id in enumerate(self.c.use_order, 1):  # 소모품 사용 (W5)
            if hero.can_act() and self.s.has_item(item_id) and self._usable(hero, self.c.items[item_id]["use"]):
                it = self.c.items[item_id]
                out.append(ChoiceView(f"use:{item_id}", f"{it['use']['label']} (남은 {self.s.inventory[item_id]}개)",
                                      [hero.id], hero.id, key=f"u{i}"))
        if self.c.rooms[self.s.room].get("rest") == "short" and self._short_rest_useful():
            out.append(ChoiceView("rest:short", f"짧은 휴식을 한다 (1시간, Hit Point Dice 남은 {hero.hd_left}개)", [], key="t"))
        if self.c.rooms[self.s.room].get("shop") and "scn-m-shop-goods" in self.s.shown:
            for i, item_id in enumerate(self.c.shop_order, 1):
                it = self.c.items[item_id]
                if self.s.stock.get(item_id) == 0 or self.s.gold_cp < it["price_cp"]:
                    continue
                out.append(ChoiceView(f"buy:{item_id}", f"구입: {it['name']} ({fmt_money(it['price_cp'])})", [], key=f"p{i}"))
        gates = self.c.rooms[self.s.room].get("exit_requires", {})  # 조건이 붙은 출구 (W7, 예: 밧줄을 고정해야 내려감)
        for i, ex in enumerate(self.c.exits(self.s.room)):
            if self.cond(gates.get(ex)):
                out.append(ChoiceView(f"move:{ex}", f"이동: {self.c.room_name(ex)}", [], key=MOVE_KEYS[i]))
        return out

    def _stabilize_choices(self, target):
        """쓰러진 동료를 돕는 선택지 (Healer's Kit / 응급 처치). 1인 플레이에서는 도울 동료가 없어 나오지 않는다."""
        out = []
        helpers = [m for m in self.s.party if m.can_act()]
        if self.s.has_item("healers_kit") and self.s.vars.get("healers_kit_uses", 0) > 0 and helpers:
            out.append(ChoiceView(f"kit:{target.id}", f"Healer's Kit로 쓰러진 {target.name}를 안정시킨다",
                                  [h.id for h in helpers], helpers[0].id, key="x1"))
        if helpers:
            best = max(helpers, key=lambda m: self.check_mod(m, {"ability": "wis", "skill": "medicine"})[0])
            out.append(ChoiceView(f"aid:{target.id}", f"쓰러진 {target.name}에게 응급 처치를 한다 (지혜(의학) DC 10)",
                                  [h.id for h in helpers], best.id, "지혜(의학) DC 10", key="x2"))
        return out

    # ------------------------------------------------------------------ 선택 실행
    def choose(self, choice_id, actor_id=None):
        """선택지 하나를 실행하고 그 결과 Event 목록을 돌려준다. 모든 행동의 단일 입구.

        지금 목록에 없는 id는 ValueError (화면이 오래된 버튼을 눌러도 규칙을 건너뛸 수 없게).
        id 접두어 → 처리 함수:
          cmb: 전투(combat.py) · hd: Hit Point Die · move: 이동 · buy: 구입 · revisit: 다시 둘러보기
          kit:/aid: 안정시키기 · wake: 깨어나기 · use: 소모품 · rest: 짧은 휴식 · inspire: 영감 재굴림
          그 밖(접두어 없음) = 장면 JSON의 선택지 → _run_choice
        처리 기록은 s.log에 남는다.
        """
        views = {v.id: v for v in self.available_choices()}
        if choice_id not in views:
            raise ValueError(f"지금 고를 수 없는 선택지: {choice_id}")
        view = views[choice_id]
        actor = None
        if view.actors:
            actor_id = actor_id or view.recommended
            if actor_id not in view.actors:
                raise ValueError(f"{actor_id}는 이 행동을 할 수 없음")
            actor = self.s.member(actor_id)
        kind, _, arg = choice_id.partition(":")
        if kind == "cmb":
            events = self.combat_choose(choice_id)
        elif kind == "hd":
            events = self._spend_hit_die() if arg == "spend" else self._finish_short_rest()
        elif kind == "move":
            events = self._enter(arg)
        elif kind == "buy":
            events = self._buy(arg)
        elif kind == "revisit":
            events = self._revisit()
        elif kind in ("kit", "aid"):
            events = self._stabilize(kind, self.s.member(arg), actor)
        elif kind == "wake":
            events = self._wake(self.s.member(arg))
        elif kind == "use":
            events = self._use_item(arg, actor)
        elif kind == "rest":
            events = self._short_rest()
        elif kind == "inspire":
            events = self._inspire(arg == "yes")
        else:
            events = self._run_choice(self.c.choices[choice_id], actor)
        self.s.log.append({"choice": choice_id, "actor": actor.id if actor else None,
                           "events": [e.text for e in events]})
        return events

    def _run_choice(self, ch, actor):
        """장면 JSON 선택지 실행: before 효과 → (check가 있으면) 판정 → success/failure 효과, (없으면) effects 효과.
        판정에 실패했는데 영웅적 영감이 있으면 결과를 적용하지 않고 pending으로 멈춰 플레이어에게 묻는다."""
        events = self._apply(ch.get("before", []), actor)  # 판정 전에 항상 일어나는 일 (NPC의 대답 등)
        if "check" in ch:
            ok, ev = self.roll_check(actor, ch["check"])
            events.append(ev)
            if not ok and actor.heroic_inspiration:  # 결과를 적용하기 전에 다시 굴릴지 묻는다 [SRD Rules Glossary]
                self.s.pending = {"type": "inspire", "choice": ch["id"], "actor": actor.id}
                events.append(Event("say", f"{actor.name}에게 영웅적 영감이 있다. 이 판정을 다시 굴릴 수 있다."))
                return events
            events += self._resolve_check(ch, actor, ok)
        else:
            events += self._apply(ch.get("effects", []), actor)
            if ch.get("once"):
                self.s.done.add(ch["id"])
        return events

    def _resolve_check(self, ch, actor, ok):
        """판정 결과를 적용한다. 성공+once → 다시 안 나옴(done), 실패 → 같은 캐릭터 재시도 금지(failed, path 판정은 예외)."""
        events = self._apply(ch["success"] if ok else ch["failure"], actor)
        if ok and ch.get("once"):
            self.s.done.add(ch["id"])
        elif not ok and not ch.get("path"):  # 길을 가는 판정(path)은 다시 시도할 수 있다 [보완 M3]
            self.s.failed.add((ch["id"], actor.id))
        return events

    def _inspire(self, use):
        """Heroic Inspiration: 주사위를 굴린 직후 다시 굴리고, 새 결과를 써야 한다 [SRD Rules Glossary].
        W5에서는 선택지의 능력 판정에만 적용 (구원·공격 굴림은 전투 구현 때)."""
        p, self.s.pending = self.s.pending, None
        ch, actor = self.c.choices[p["choice"]], self.s.member(p["actor"])
        if not use:
            return self._resolve_check(ch, actor, False)
        actor.heroic_inspiration = False
        ok, ev = self.roll_check(actor, ch["check"])
        return [Event("say", f"{actor.name}가 영웅적 영감을 써서 다시 굴린다."), ev] + self._resolve_check(ch, actor, ok)

    # ------------------------------------------------------------------ 효과 (05-2 5장)
    def _apply(self, effects, actor):
        """효과 목록을 차례로 실행해 상태를 바꾸고 Event를 모은다. 장면 JSON의 effects/success/failure 등이 여기로 온다.

        효과는 키 하나로 종류가 정해지는 dict이다. 예: {"flag": "tracked_rats"}, {"give": "rope"},
        {"show": 카드 id}, {"combat": 조우 id, "on_victory": [...]}, {"if": 조건, "then": [...], "else": [...]}.
        새 효과 종류는 아래 elif에 추가한다 (모르는 키는 ValueError). 게임 오버가 되면 남은 효과는 실행하지 않는다.
        """
        events = []
        for e in effects:
            if self.s.ended == "game_over":
                break
            if "use" in e:
                events += self._apply(self.c.effect_sets[e["use"]], actor)
            elif "enter" in e:  # 다른 방(다른 장면 포함)으로 이동 [W3]
                events += self._enter(e["enter"])
            elif "if" in e:
                events += self._apply(e["then"] if self.cond(e["if"], actor) else e.get("else", []), actor)
            elif "show" in e:
                events += self._show(e["show"], actor)
            elif "flag" in e:
                if e["flag"] not in self.s.flags:
                    self.s.flags.add(e["flag"])
                    events.append(Event("flag", f"(상태: {e['flag']})", {"flag": e["flag"]}))
            elif "var" in e:
                self.s.vars[e["var"]] = e["value"]
            elif "give" in e:
                self.s.inventory[e["give"]] = self.s.inventory.get(e["give"], 0) + e.get("n", 1)
                events.append(Event("item", f"획득: {self.c.items[e['give']]['name']}", {"item": e["give"]}))
            elif "gold_cp" in e:
                self.s.gold_cp += e["gold_cp"]
                events.append(Event("gold", f"돈 {fmt_money(e['gold_cp'])} 획득 (소지금 {fmt_money(self.s.gold_cp)})"))
            elif "say" in e:
                events.append(Event("say", e["say"]))
            elif "trap" in e:
                events += self._trap(e["trap"], actor)
            elif "disarm_trap" in e:
                self.s.traps.setdefault(e["disarm_trap"], {"count": 0, "disarmed": False})["disarmed"] = True
            elif "add_condition" in e:
                (actor or self.hero).conditions[e["add_condition"]] = e.get("minutes")
            elif "remove_condition" in e:
                (actor or self.hero).conditions.pop(e["remove_condition"], None)
            elif "inspiration" in e:
                events += self._gain_inspiration(actor or self.hero)
            elif "combat" in e:  # 전투 시작 (W6): {"combat": 조우 id, "on_victory": [...], "on_retreat": [...]}
                on_end = {k[3:]: v for k, v in e.items() if k.startswith("on_")}
                events += self.start_combat(e["combat"], on_end)
            elif "level_up" in e:
                events += self._level_up(actor or self.hero, e["level_up"])
            elif "long_rest" in e:
                events += self._long_rest(e["long_rest"]["cost_cp_per_person"])
            elif "fall" in e:  # 낙하 (W7): d6 × 10ft, 이 구간의 높이까지만 [보완], 10ft당 1d6 [SRD Falling]
                events += self._fall(actor or self.hero, e["fall"])
            elif "end_scene" in e:
                self.s.ended = f"scene{self.s.scene}_complete"
                events.append(Event("end", e["end_scene"]))
            else:
                raise ValueError(f"알 수 없는 효과: {e}")
        return events

    # ------------------------------------------------------------------ 함정 (W5: traps.json 데이터로 처리)
    def trap_active(self, trap_id):
        """해제되지 않았고, 발동 횟수가 한도 미만이면 작동한다.
        한도: 지속 시간에 재장전이 없으면 1회 [SRD p.199], 독침 다트는 3회 [SRD p.200]."""
        st = self.s.traps.get(trap_id, {"count": 0, "disarmed": False})
        limit = self.c.traps[trap_id].get("max_triggers")
        return not st["disarmed"] and (limit is None or st["count"] < limit)

    def resists(self, char, damage_type):
        """종족 특성으로 이 피해 종류에 저항하는지 (드워프 → 독). 함정·몬스터 공격 피해에서 절반 처리에 쓴다."""
        return any(damage_type in TRAIT_RESIST.get(t, ()) for t in char.traits)

    def _trap(self, trap_id, actor):
        """함정 발동 ({"trap": id} 효과). 수치는 모두 traps.json에서 읽는다.
        순서: 발동 횟수 +1 → 발동 문장 → 구원 굴림 → 실패 시 상태 → 피해(성공 시 절반·저항 절반) → 뒤 문장·소음 기록."""
        t = self.c.traps[trap_id]
        victim = actor or self.hero
        if not self.trap_active(trap_id):
            return []
        self.s.traps.setdefault(trap_id, {"count": 0, "disarmed": False})["count"] += 1
        events = [Event("say", t["trigger_text"], {"trap": trap_id})]
        save = t.get("save")
        ok = False
        if save:
            ok, ev = self.roll_save(victim, save["ability"], save["dc"], avoids=save.get("avoids"))
            events.append(ev)
        fc = t.get("fail_condition")
        if fc and not ok:  # 상태는 맞는 순간 걸린다 (쓰러져 시간이 흐르면 그만큼 줄어듦)
            victim.conditions[fc["name"]] = fc["minutes"]
        if "damage" in t and not (ok and t.get("on_save_success") == "none"):
            n, why = 1, ""
            if "damage_count" in t:  # 예: 다트 1d3개, 개당 1d6 [원본 scn-g-darts-trap]
                n, _ = self.dice.roll(t["damage_count"])
                why = f"{n}개 × "
            rolls = []
            for _ in range(n):
                rolls += self.dice.roll(t["damage"])[1]
            dmg = sum(rolls)
            dtype = t.get("damage_type", "")
            why += f"{DAMAGE_KO.get(dtype, dtype)} 피해 {t['damage']}({'+'.join(map(str, rolls))})"
            if ok:  # 성공 시 절반, 내림 [SRD p.17]
                dmg //= 2
                why += " 절반"
            if self.resists(victim, dtype):
                dmg //= 2
                why += " · 저항"
            events += self._damage(victim, dmg, why)
        if fc and not ok and not victim.down and t.get("fail_condition_text") and fc["name"] in victim.conditions:
            events.append(Event("status", t["fail_condition_text"].format(name=victim.name)))
        if t.get("after_text") and self.s.ended != "game_over":
            events.append(Event("say", t["after_text"]))
        if t.get("noise"):
            self.s.vars[f"noise:{trap_id}"] = True  # 소음 → 몬스터 반응은 해당 장면에서 처리 [원본]
        return events

    def _fall(self, char, spec):
        """[원본 scn-s2-check-descend] 실패하면 d6 × 10피트 낙하, 10피트당 1d6 (최대 20d6) [SRD Falling].
        [보완 W7] 떨어지는 거리는 그 구간의 높이(max_ft)를 넘지 않는다."""
        n, _ = self.dice.roll("1d6")
        feet = min(n * 10, spec.get("max_ft", 200))
        dice = min(20, feet // 10)
        dmg, rolls = self.dice.roll(f"{dice}d6")
        events = [Event("say", f"손이 미끄러진다! {feet}피트 아래로 떨어졌다.", {"fall_ft": feet})]
        return events + self._damage(char, dmg, f"낙하 {feet}피트 {dice}d6({'+'.join(map(str, rolls))})")

    # ------------------------------------------------------------------ 소모품 (W5)
    def _usable(self, char, use):
        """소모품을 지금 쓰는 의미가 있는지 (HP가 가득이면 물약 선택지를 숨긴다 등). use = items.json의 "use" 값."""
        if "heal" in use:
            return char.hp < char.max_hp
        if "condition" in use:
            return use["condition"] not in char.conditions
        return True

    def _use_item(self, item_id, user):
        """전투 밖에서는 시간 비용 없이 바로 사용 (보너스 행동은 전투 구현 W6에서 행동 경제에 연결)."""
        it, use = self.c.items[item_id], self.c.items[item_id]["use"]
        self.s.inventory[item_id] -= 1
        events = [Event("item", f"{user.name}가 {it['name']}을(를) 사용했다. (남은 {self.s.inventory[item_id]}개)", {"item": item_id})]
        if "heal" in use:
            amount, rolls = self.dice.roll(use["heal"])
            before = user.hp
            user.hp = min(user.max_hp, user.hp + amount)
            events.append(Event("status", f"{use['heal']}({'+'.join(map(str, rolls))}) = {amount} 회복. "
                                          f"HP {before} → {user.hp}/{user.max_hp}", {"heal": user.hp - before}))
        if "condition" in use:
            user.conditions[use["condition"]] = use["minutes"]
            events.append(Event("status", f"{user.name}는 {use['minutes'] // 60}시간 동안 중독을 피하거나 끝내는 구원 굴림에 이점을 얻는다."))
        return events

    # ------------------------------------------------------------------ 짧은 휴식·Hit Point Dice (W5)
    def _hd_useful(self, m):
        """Hit Point Die를 쓸 수 있고 쓸 이유가 있는지 (의식이 있고, 다쳤고, 주사위가 남음)."""
        return m.hp >= 1 and m.hp < m.max_hp and m.hd_left > 0

    def _recharge_pending(self, m):
        """짧은 휴식으로 회복할 직업 기능이 남아 있는지 (Second Wind 1회, Action Surge, Channel Divinity 1회, Arcane Recovery)."""
        full = self.c.rules["class_levels"].get(m.cls, {}).get(str(m.level), {})
        for k in ("second_wind", "action_surge", "channel_divinity"):
            if k in full and m.resources.get(k, 0) < full[k]:
                return True
        slots = m.resources.get("slots")
        return bool(m.resources.get("arcane_recovery") and slots and slots != full.get("slots"))

    def _short_rest_useful(self):
        """짧은 휴식 선택지를 보여 줄지. 회복할 것이 없으면 숨긴다 (휴식 가능한 방 = 방 데이터의 "rest": "short")."""
        return any(m.hp >= 1 and (self._hd_useful(m) or self._recharge_pending(m)) for m in self.s.party)

    def _short_rest(self):
        """[SRD Rules Glossary: Short Rest] 1시간, HP 1 이상이어야 시작. 끝날 때 Hit Point Dice를 원하는 만큼 쓴다:
        주사위 하나마다 굴린 값 + 건강 수정치(최소 1), 한 개씩 굴리고 더 쓸지 정한다 (W6: 플레이어가 고름)."""
        events = self._pass_time(SHORT_REST_MIN)
        events.append(Event("rest", "잠시 숨을 돌리며 상처를 살핀다. (짧은 휴식 1시간)"))
        if self._hd_useful(self.hero):
            self.s.pending = {"type": "hit_dice"}
            events.append(Event("status", f"Hit Point Dice를 몇 개 쓸지 고른다. (남은 {self.hero.hd_left}개, d{self.hero.hit_die})"))
            return events
        return events + self._finish_short_rest()

    def _spend_hit_die(self):
        """'hd:spend' — Hit Point Die 1개를 굴려 회복. 더 쓸 수 없게 되면 휴식을 자동으로 끝낸다."""
        m = self.hero
        r, _ = self.dice.roll(f"1d{m.hit_die}")
        gain = max(1, r + m.mod("con"))
        twice = ""
        if self.s.has_item(PERIAPT_WOUND):  # [원본 scn-h-fighter-periapt] Hit Point Die 회복량 두 배
            gain, twice = gain * 2, " ×2 (Periapt)"
        m.hd_left -= 1
        before, m.hp = m.hp, min(m.max_hp, m.hp + gain)
        events = [Event("status", f"{m.name}가 Hit Point Die를 썼다: d{m.hit_die}({r}) {m.mod('con'):+d}{twice} = {gain} 회복. "
                                  f"HP {before} → {m.hp}/{m.max_hp} (남은 주사위 {m.hd_left}/{m.hd_max})")]
        if not self._hd_useful(m):  # 더 쓸 주사위가 없거나 HP가 가득 차면 휴식을 마친다
            events += self._finish_short_rest()
        return events

    def _finish_short_rest(self):
        """짧은 휴식이 끝날 때 회복되는 기능 [SRD p.37·47·78]."""
        self.s.pending = None
        events = [Event("rest", "짧은 휴식을 마쳤다.")]
        for m in self.s.party:
            full = self.c.rules["class_levels"].get(m.cls, {}).get(str(m.level), {})
            r = m.resources
            if "second_wind" in full and r.get("second_wind", 0) < full["second_wind"]:
                r["second_wind"] += 1                 # 1회 회복 [SRD p.47]
                events.append(Event("status", f"Second Wind 1회 회복 (남은 {r['second_wind']}회)"))
            if "action_surge" in full and r.get("action_surge", 0) < full["action_surge"]:
                r["action_surge"] = full["action_surge"]   # [SRD p.47]
                events.append(Event("status", "Action Surge 회복"))
            if "channel_divinity" in full and r.get("channel_divinity", 0) < full["channel_divinity"]:
                r["channel_divinity"] += 1            # 1회 회복 [SRD p.37]
                events.append(Event("status", f"Channel Divinity 1회 회복 (남은 {r['channel_divinity']}회)"))
            if r.get("arcane_recovery") and r.get("slots") and r["slots"] != full.get("slots"):
                # Arcane Recovery: 합계 레벨 ≤ 마법사 레벨의 절반(올림), 긴 휴식마다 1회 [SRD p.78]
                # 쓴 슬롯이 있으면 자동 사용, 높은 슬롯부터 [보완]
                budget = math.ceil(m.level / 2)
                got = []
                for lv in range(len(r["slots"]), 0, -1):
                    while budget >= lv and r["slots"][lv - 1] < full["slots"][lv - 1]:
                        r["slots"][lv - 1] += 1
                        budget -= lv
                        got.append(lv)
                if got:
                    r["arcane_recovery"] = 0
                    events.append(Event("status", f"Arcane Recovery: 주문 슬롯 회복 {got} (남은 슬롯 {r['slots']})"))
        return events

    # ------------------------------------------------------------------ 영웅적 영감 (W5)
    def _gain_inspiration(self, char):
        """이미 있으면 새로 얻은 것은 다른 PC에게 줄 수 있다. 1인이라 줄 사람이 없으므로 사라진다 [SRD Rules Glossary]."""
        if char.heroic_inspiration:
            return [Event("status", f"{char.name}는 이미 영웅적 영감을 가지고 있다. (새로 얻은 영감은 사라짐)")]
        char.heroic_inspiration = True
        return [Event("status", f"{char.name}가 영웅적 영감을 얻었다. 실패한 판정을 한 번 다시 굴릴 수 있다.")]

    # ------------------------------------------------------------------ 피해·0 HP [SRD p.17-18] + [보완 05-6 ②]
    def _damage(self, char, amount, why, critical=False):
        """캐릭터가 피해를 받는 유일한 통로 (함정·낙하·몬스터 공격 모두 여기로 온다).

        HP가 0이 되면: 남은 피해 ≥ 최대 HP → 즉사, 아니면 dying(빈사) 상태가 되고 _after_dropping으로 넘어간다.
        이미 0 HP일 때 맞으면 죽음 구원 실패가 1회(치명타 2회) 늘어난다.
        """
        events = []
        if "dead" in char.conditions:
            return events
        if char.hp == 0:  # 이미 0 HP에서 피해 [SRD p.18]
            events.append(Event("damage", f"{char.name}가 쓰러진 채로 {amount} 피해를 입었다 ({why})."))
            if amount >= char.max_hp:
                return events + self._die(char, "쓰러진 상태에서 최대 HP 이상의 피해")
            char.conditions.pop("stable", None)
            st = char.conditions.setdefault("dying", {"success": 0, "fail": 0})
            st["fail"] += 2 if critical else 1
            if st["fail"] >= 3:
                return events + self._die(char, "죽음 구원 실패 3회")
            return events + self._after_dropping(char)
        remainder = amount - char.hp
        char.hp = max(0, char.hp - amount)
        events.append(Event("damage", f"{char.name}가 {amount} 피해를 입었다 ({why}). HP {char.hp}/{char.max_hp}",
                            {"char": char.id, "amount": amount}))
        events += self._concentration_check(char, amount)  # 집중 유지 [SRD 용어집 Concentration]
        if char.hp > 0:
            return events
        if remainder >= char.max_hp:  # 즉사: 남은 피해 ≥ 최대 HP [SRD p.17]
            return events + self._die(char, f"초과 피해 {remainder} ≥ 최대 HP {char.max_hp}")
        char.conditions["dying"] = {"success": 0, "fail": 0}
        events.append(Event("status", f"{char.name}가 쓰러져 의식을 잃었다!"))
        return events + self._after_dropping(char)

    def _after_dropping(self, char):
        """도와줄 동료가 없으면 죽음 구원을 결과가 날 때까지 자동으로 굴린다 [보완 05-6 D1].
        (전투 밖이므로 '턴'이 없어 1회 = 1턴으로 본다. 전투 중에는 자기 턴마다 1회 — combat.py D2·D4)"""
        if self.s.combat:
            return []
        helpers = [m for m in self.s.party if m is not char and m.can_act()]
        if helpers:
            return []
        events = []
        while "dying" in char.conditions:
            events += self._death_save(char)
        if "stable" in char.conditions:
            events += self._wake(char)
        return events

    def _death_save(self, char):
        """죽음 구원 1회 [SRD p.17-18]: 10 이상 성공, 1은 실패 2회, 20은 1 HP 회복."""
        st = char.conditions["dying"]
        d, _ = self.dice.d20()
        if d == 20:
            char.conditions.pop("dying")
            char.hp = 1
            return [Event("check", f"{char.name}의 죽음 구원 굴림: d20 20 → 기적처럼 의식을 되찾았다! HP 1/{char.max_hp}")]
        note = ""
        if d <= 9 and self.s.has_item(PERIAPT_WOUND):
            # [원본 scn-h-fighter-periapt] 착용하면 죽음 구원 굴림에서 9 이하가 나와도 10으로 바꿀 수 있다
            note, d = f" (Periapt of Wound Closure: {d} → 10)", 10
        if d == 1:
            st["fail"] += 2
            result = "실패 2회 (1)"
        elif d >= 10:
            st["success"] += 1
            result = "성공"
        else:
            st["fail"] += 1
            result = "실패"
        events = [Event("check", f"{char.name}의 죽음 구원 굴림: d20 {d}{note} → {result} (성공 {st['success']} / 실패 {st['fail']})")]
        if st["fail"] >= 3:
            events += self._die(char, "죽음 구원 실패 3회")
        elif st["success"] >= 3:
            events += self._make_stable(char)
        return events

    def _die(self, char, why):
        """사망 처리. 파티 전원이 죽으면 s.ended = "game_over" (화면은 이 값을 보고 게임 오버 창을 띄운다)."""
        char.conditions.pop("dying", None)
        char.conditions.pop("stable", None)
        char.conditions["dead"] = None
        events = [Event("status", f"{char.name}가 숨을 거두었다. ({why})")]
        if all("dead" in m.conditions for m in self.s.party):
            self.s.ended = "game_over"
            events.append(Event("end", f"게임 오버. 체크포인트({self.checkpoint_label})에서 다시 시작할 수 있다."))
        return events

    def _make_stable(self, char):
        char.conditions.pop("dying", None)
        char.conditions["stable"] = None
        return [Event("status", f"{char.name}가 안정되었다. 의식은 없지만 더는 위험하지 않다.")]

    def _stabilize(self, kind, target, helper):
        """동료가 쓰러진 캐릭터를 돕는다 (kit: Healer's Kit / aid: 지혜(의학) DC 10). 먼저 그 턴의 죽음 구원을 굴린다."""
        events = self._death_save(target)
        if "dying" not in target.conditions:
            return events
        if kind == "kit":
            # [원본 scn-g-chest-loot] Healer's Kit: 판정 없이 안정
            self.s.vars["healers_kit_uses"] -= 1
            events.append(Event("say", f"{helper.name}가 Healer's Kit로 상처를 묶는다. (남은 사용 {self.s.vars['healers_kit_uses']}회)"))
            return events + self._make_stable(target)
        ok, ev = self.roll_check(helper, {"ability": "wis", "skill": "medicine", "dc": 10})
        events.append(ev)
        if ok:
            events += self._make_stable(target)
        return events

    def _wake(self, char):
        hours, _ = self.dice.roll("1d4")  # 치료받지 않은 안정 상태는 1d4시간 뒤 1 HP [SRD p.18]
        events = self._pass_time(hours * 60)
        char.conditions.pop("stable", None)
        char.hp = 1
        events.append(Event("status", f"{hours}시간이 지나 {char.name}가 깨어났다. HP 1/{char.max_hp}"))
        return events

    # ------------------------------------------------------------------ 체크포인트 [보완 05-6 D5]
    def _save_checkpoint(self, label):
        """장면 시작 또는 긴 휴식 직후. 둘 중 더 최근 것이 남는다 (덮어쓰기)."""
        self.checkpoint = copy.deepcopy(self.s)
        self.checkpoint_label = label

    def restart_from_checkpoint(self):
        """게임 오버 뒤 '다시 시작'. 저장해 둔 GameState 복사본으로 통째로 되돌린다 (주사위 상태는 되돌리지 않음)."""
        if self.checkpoint is None:
            raise RuntimeError("체크포인트가 없음")
        self.s = copy.deepcopy(self.checkpoint)
        return [Event("move", f"== 체크포인트({self.checkpoint_label})에서 다시 시작: {self.c.room_name(self.s.room)} ==",
                      {"room": self.s.room})]

    # ------------------------------------------------------------------ 시간·휴식·이동·상점
    def _pass_time(self, minutes):
        """게임 시간을 흘려보낸다. conditions 값이 정수(남은 분)인 상태만 줄어들고, 0 이하가 되면 풀린다.
        값이 None이나 dict인 상태(dying, stable, grappled 등)는 시간으로 끝나지 않는다."""
        self.s.minutes += minutes
        events = []
        for m in self.s.party:  # 시간이 정해진 상태·효과(분 단위 정수)만 줄어든다
            for name, left in list(m.conditions.items()):
                if type(left) is not int:
                    continue
                m.conditions[name] = left - minutes
                if m.conditions[name] <= 0:
                    del m.conditions[name]
                    if m.concentration == name:  # 지속 시간이 끝난 집중 주문
                        m.concentration = ""
                    events.append(Event("status", EXPIRE_TEXT.get(name, "{name}의 " + name + " 상태가 끝났다.").format(name=m.name)))
        return events

    def _long_rest(self, cost_per_person):
        """여관 긴 휴식 ({"long_rest": …} 효과, 장면 1). 숙박비 지불 → 8시간 → HP·Hit Point Dice·직업 자원 전부 회복
        → 체크포인트 저장. 16시간 안에 다시 하거나 돈이 모자라면 문장만 돌려주고 아무것도 바꾸지 않는다."""
        s = self.s
        last = s.vars.get("last_long_rest_end")
        if last is not None and s.minutes - last < LONG_REST_GAP_MIN:
            return [Event("say", "긴 휴식을 마친 지 16시간이 지나지 않아 다시 긴 휴식을 할 수 없다.")]
        cost = cost_per_person * len([m for m in s.party if "dead" not in m.conditions])
        if s.gold_cp < cost:
            return [Event("say", f"숙박비({fmt_money(cost)})가 부족하다.")]
        s.gold_cp -= cost
        events = [Event("gold", f"숙박비 {fmt_money(cost)} 지불 (소지금 {fmt_money(s.gold_cp)})")]
        events += self._pass_time(LONG_REST_MIN)
        for m in s.party:
            if m.hp >= 1:  # 긴 휴식은 HP 1 이상이어야 시작할 수 있다 [SRD Rules Glossary]
                m.ability_loss = {}   # 줄어든 능력치 복구 [SRD Rules Glossary: Long Rest]
                m.hp_max_loss = 0     # Specter 생명 흡수: 회복 시점은 SRD 블록에 없음 → 긴 휴식에서 복구 [보완]
                m.hp = m.hp_max
                m.hd_left = m.hd_max  # 쓴 Hit Point Dice를 모두 회복 [SRD Rules Glossary: Long Rest]
                self.init_resources(m)  # 주문 슬롯·Second Wind 등 모두 회복 [SRD 직업 기능]
        s.vars["last_long_rest_end"] = s.minutes
        events.append(Event("rest", "여관에서 하룻밤을 보냈다. (긴 휴식: HP와 Hit Point Dice를 모두 회복) 종소리는 밤새 이어졌다."))
        self._save_checkpoint("긴 휴식 직후")
        events.append(Event("status", "(체크포인트 저장: 긴 휴식 직후)"))
        return events

    def _enter(self, room_id):
        """방에 들어간다 (move: 선택지, {"enter": 방} 효과, 게임 시작). 다른 장면의 방이면 장면 전환도 여기서 일어난다.

        순서: 위치·방문 기록·장면 번호 갱신 → (새 장면이면) 장면 전환 Event → 방 이름 Event
        → 그 방의 장면 묘사(desc) 카드 표시 → 방 진입 반응(on_enter, 한 번만) → (새 장면이면) 체크포인트 저장.
        """
        s = self.s
        if room_id not in self.c.rooms:
            raise ValueError(f"불러온 장면에 없는 방: {room_id}")
        prev_scene = s.scene
        new_scene = self.c.rooms[room_id]["scene"] > s.scene
        s.room = room_id
        s.visited.add(room_id)
        s.scene = max(s.scene, self.c.rooms[room_id]["scene"])  # 장면 번호는 올라가기만 함 [원본 p.3]
        events = []
        if new_scene:
            events.append(Event("scene", f"=== 장면 {prev_scene} 완료 → 장면 {s.scene}: {self.c.scene_titles[s.scene]} ===",
                                {"from": prev_scene, "to": s.scene}))
        events.append(Event("move", f"== {self.c.room_name(room_id)} ==", {"room": room_id}))
        for cid, card in self.c.cards.items():  # 방에 들어가면 장면 묘사 카드 표시
            if card["room"] == room_id and card["kind"] == "desc":
                events += self._show(cid)
        for i, hook in enumerate(self.c.rooms[room_id].get("on_enter", [])):  # 방 진입 반응 [보완]
            key = f"enter:{room_id}:{i}"
            if key not in s.once_done and self.cond(hook.get("requires")):
                s.once_done.add(key)
                events += self._apply(hook["effects"], None)
        if new_scene:
            if self.c.scene_status.get(s.scene) == "skeleton":
                events.append(Event("say", f"(장면 {s.scene}의 콘텐츠는 아직 구현되지 않았습니다. 방 이동만 가능합니다.)"))
            self._save_checkpoint(f"장면 {s.scene} 시작")
            events.append(Event("status", f"(체크포인트 저장: 장면 {s.scene} 시작)"))
        return events

    def _buy(self, item_id):
        """상점 구입 (buy: 선택지). 살 수 있는지(돈·재고)는 available_choices에서 이미 걸렀으므로 여기서는 바로 처리한다."""
        it = self.c.items[item_id]
        self.s.gold_cp -= it["price_cp"]
        if self.s.stock.get(item_id) is not None:
            self.s.stock[item_id] -= 1
        self.s.inventory[item_id] = self.s.inventory.get(item_id, 0) + 1
        if item_id == "healers_kit":
            self.s.vars["healers_kit_uses"] = self.s.vars.get("healers_kit_uses", 0) + it["uses"]
        events = [Event("item", f"{it['name']}을(를) 샀다. (소지금 {fmt_money(self.s.gold_cp)})", {"item": item_id})]
        reaction = self.c.buy_reactions.get(item_id)  # 구매 반응 [보완]
        if reaction and f"buy:{reaction['id']}" not in self.s.once_done and self.cond(reaction.get("requires")):
            self.s.once_done.add(f"buy:{reaction['id']}")
            events += self._apply(reaction["effects"], None)
        return events

In [ ]:
def fmt_money(cp):
    """cp 단위 정수를 화면용 글자로. 예: 1250 → '12 gp 5 sp'. 돈은 항상 cp 정수로 저장한다."""
    gp, rest = divmod(cp, 100)
    sp, cp_ = divmod(rest, 10)
    parts = [f"{gp} gp" if gp else "", f"{sp} sp" if sp else "", f"{cp_} cp" if cp_ else ""]
    return " ".join(p for p in parts if p) or "0 gp"

## [7] AI 서술기 (선택 기능, 기본은 꺼짐)
원본 파일: `game/narrator.py`

```text
AI 마스터 서술 (Qwen). 판정·상태는 엔진이 이미 끝낸 뒤, 결과를 문장으로만 바꾼다.
정보 접근 원칙 (analysis/05_시스템설계/05-2a 3장):
  Qwen에게는 이번 행동으로 '플레이어에게 표시된' 카드 문장, 판정 결과, 짧은 상태 요약만 넘긴다.
  열람 가능하지만 표시되지 않은 정보(함정, 비밀 카드)는 넘기지 않는다.
백엔드:
  - TemplateNarrator: LLM 없이 진행 (서술 없음, 엔진 문장만 출력)
  - OllamaNarrator:   기존 Colab 노트북(TRPG_Rule.ipynb)과 같은 Ollama /api/chat 호출 (예: qwen2.5:7b)
  - CallableNarrator: Colab에서 transformers로 띄운 Qwen 등, messages -> str 함수를 직접 연결
세 클래스 모두 narrate(engine, events, actor_id) 하나만 가진다. 돌려준 문장은 화면에 덧붙여 보여 줄 뿐,
게임 상태에는 아무 영향이 없다. 쓰는 곳: game/cli.py play(), server/webapp.py Game.reply().
```

**들어 있는 것**
- **`build_messages`** (함수) — 엔진 이벤트 → Qwen 입력. 표시된 카드(card 이벤트)와 결과 문장만 담는다.
- **`clean`** (함수) — 모델이 붙이는 <think>…</think> 생각 과정을 지우고 앞뒤 공백을 정리한다.
- **`TemplateNarrator`** (클래스) — (설명 없음)
- **`CallableNarrator`** (클래스) — (설명 없음)
- **`OllamaNarrator`** (클래스) — (설명 없음)

In [ ]:
SYSTEM_PROMPT = """너는 한국어 TRPG의 진행자(마스터)다. 아래 규칙을 반드시 지킨다.
1. 판정 결과(성공/실패), 피해 숫자, 획득한 물건은 [결과]에 적힌 그대로만 말한다. 바꾸거나 새로 만들지 않는다.
2. [이번에 드러난 정보]와 [결과]에 없는 사실(숨겨진 함정, 비밀, 다른 장소의 정보)을 지어내거나 암시하지 않는다.
3. 플레이어가 다음에 무엇을 해야 하는지 지시하지 않는다.
4. 2~4문장, 현재형 서술. 주사위 숫자나 DC는 다시 말하지 않는다.
5. 전투에서 명중·빗나감·피해·쓰러짐·승패는 [결과]에 적힌 그대로만 묘사한다. 결과에 없는 공격, 추가 피해, 적의 죽음이나 도주를 만들지 않는다."""


THINK_TAG = re.compile(r"<think>.*?</think>", re.S)


def build_messages(engine, events, actor_id=None):
    """엔진 이벤트 → Qwen 입력. 표시된 카드(card 이벤트)와 결과 문장만 담는다."""
    s = engine.s
    revealed = [e.text for e in events if e.kind == "card"]
    results = [e.text for e in events if e.kind in ("check", "say", "damage", "item", "gold", "status", "rest", "end", "combat")]
    actor = s.member(actor_id).name if actor_id else "일행"
    party = ", ".join(f"{m.name}({m.cls_ko}, HP {m.hp}/{m.max_hp})" for m in s.party)
    user = (f"[장소] {engine.c.room_name(s.room)}\n[파티] {party}\n[행동한 사람] {actor}\n"
            f"[이번에 드러난 정보]\n" + ("\n".join(revealed) or "(없음)") +
            "\n[결과]\n" + ("\n".join(results) or "(특별한 결과 없음)") +
            "\n\n위 내용만 바탕으로 장면을 묘사해 줘.")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def clean(text):
    """모델이 붙이는 <think>…</think> 생각 과정을 지우고 앞뒤 공백을 정리한다."""
    return THINK_TAG.sub("", text or "").strip()

In [ ]:
class TemplateNarrator:
    name = "template"

    def narrate(self, engine, events, actor_id=None):
        return None  # 엔진 문장을 그대로 보여 준다

In [ ]:
class CallableNarrator:
    name = "callable"

    def __init__(self, fn):
        self.fn = fn

    def narrate(self, engine, events, actor_id=None):
        try:
            return clean(self.fn(build_messages(engine, events, actor_id)))
        except Exception as e:  # 서술 실패가 게임 진행을 막지 않게 한다
            return f"(서술 생성 실패: {e})"

In [ ]:
class OllamaNarrator:
    name = "ollama"

    def __init__(self, base_url="http://localhost:11434", model="qwen2.5:7b", timeout=60, temperature=0.4):
        self.url = base_url.rstrip("/") + "/api/chat"
        self.model = model
        self.timeout = timeout
        self.temperature = temperature

    def narrate(self, engine, events, actor_id=None):
        body = json.dumps({
            "model": self.model,
            "messages": build_messages(engine, events, actor_id),
            "stream": False,
            "options": {"temperature": self.temperature},
        }).encode("utf-8")
        req = urllib.request.Request(self.url, data=body, headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                data = json.loads(r.read().decode("utf-8"))
            return clean(data["message"]["content"])
        except (urllib.error.URLError, OSError, KeyError, ValueError) as e:
            return f"(서술 생성 실패: {e})"

## [8] 터미널(글자) 화면
원본 파일: `game/cli.py`

```text
터미널에서 플레이한다 (기획서: 1~2주차는 터미널 화면으로 대체 가능). 장면 파일을 모두 이어서 진행.
  python -m game.cli                         # 시작할 때 캐릭터를 고름, LLM 없이 진행
  python -m game.cli --char harin --seed 7   # 캐릭터·주사위 시드 지정
  python -m game.cli --narrator ollama       # Ollama 서술 (모델은 --model, 고정하지 않음)
웹 화면(server/ + web/)과 같은 엔진을 쓰는 글자 화면이다. 규칙 처리는 하지 않고,
engine.available_choices()를 출력 → 키 입력 → engine.choose()를 반복할 뿐이다.
tools/play_script.py와 Colab 셀 [16]이 play()를 그대로 다시 쓴다.
```

**들어 있는 것**
- **`print_events`** (함수) — 엔진 Event를 한 줄씩 출력 (flag 기록은 내부 정보라 숨김).
- **`status`** (함수) — 's' 입력 — 위치·HP·상태·소지품 요약.
- **`known`** (함수) — 'k' 입력 — 지금까지 알게 된 카드 목록.
- **`pick_actor`** (함수) — (설명 없음)
- **`choose_character`** (함수) — (설명 없음)
- **`play`** (함수) — 게임 반복문. ask는 입력 함수라서, 테스트·play_script는 input 대신 정해진 답을 돌려주는 함수를 넣는다.
- **`cli_main`** (함수) — (설명 없음)

In [ ]:
HELP = ("입력 방법: 숫자 = 이 방의 행동, 영문자 a~e = 이동, p숫자 = 구입, u숫자 = 소모품 사용, t = 짧은 휴식, "
        "r = 새 소식 듣기, i/n = 영웅적 영감으로 다시 굴리기/그대로 두기, "
        "s = 상태, k = 알게 된 정보, h = 도움말, q = 종료\n"
        "  · 전투 중: 숫자 = 행동(공격·주문·회피 등), b숫자 = 보너스 행동, x숫자 = 물러나기 등, v = Action Surge, z = 차례 마침\n"
        "  · 같은 행동은 턴이 바뀌어도 같은 키를 씁니다 (끝난 행동은 목록에서 빠지고, 번호는 당겨지지 않습니다).\n"
        "  · 자유 문장 입력(예: '줄을 당겨본다')은 아직 지원하지 않습니다. 목록의 키를 입력하세요.")


LINE = "─" * 60


def print_events(events):
    """엔진 Event를 한 줄씩 출력 (flag 기록은 내부 정보라 숨김)."""
    for e in events:
        if e.kind == "flag":
            continue
        if e.kind == "scene":
            print(f"\n{e.text}")
            continue
        print(("  " if e.kind != "move" else "") + e.text)


def status(engine):
    """'s' 입력 — 위치·HP·상태·소지품 요약."""
    s = engine.s
    print(f"-- 장면 {s.scene} · {engine.c.room_name(s.room)} · 소지금 {fmt_money(s.gold_cp)} · 경과 {s.minutes // 60}시간"
          f" · 체크포인트: {engine.checkpoint_label}")
    for m in s.party:
        cond = ", ".join(k for k in m.conditions) or "-"
        insp = " · 영웅적 영감 있음" if m.heroic_inspiration else ""
        print(f"   {m.name} ({m.cls_ko} {m.level}레벨, {m.species}) HP {m.hp}/{m.max_hp}  AC {engine.armor_class(m)}"
              f"  Hit Point Dice {m.hd_left}/{m.hd_max}(d{m.hit_die})  상태: {cond}{insp}")
        if m.resources:
            print(f"   남은 사용 횟수: {m.resources}")
    if s.combat:
        foes = ", ".join(f"{x['name']}({engine.wound_text(x)})" for x in engine.living_enemies())
        print(f"   전투 중 ({s.combat['round']}라운드): {foes}")
    inv = ", ".join(f"{engine.c.items[i]['name']}×{n}" for i, n in s.inventory.items() if n) or "-"
    print(f"   소지품: {inv}")


def known(engine):
    """'k' 입력 — 지금까지 알게 된 카드 목록."""
    for cid in engine.player_known_cards():
        card = engine.c.cards[cid]
        print(f"  - [{card['label']}] {engine._view_text(cid)[0][:70]}…")


def pick_actor(engine, view, ask):
    # 보완 A: 판정이 없는 행동은 묻지 않고 추천 캐릭터가 한다. (1인에서는 항상 그 캐릭터)
    if len(view.actors) <= 1 or not view.check_text:
        return view.recommended or (view.actors[0] if view.actors else None)
    names = {m.id: m.name for m in engine.s.party}
    opts = " / ".join(f"{i + 1}.{names[a]}" for i, a in enumerate(view.actors))
    try:
        ans = ask(f"   누가 할까요? {opts} (Enter = 추천: {names[view.recommended]}) > ").strip().lstrip("﻿")
    except EOFError:
        ans = ""
    if ans.isdigit() and 1 <= int(ans) <= len(view.actors):
        return view.actors[int(ans) - 1]
    return view.recommended


def choose_character(content, ask):
    chars = content.characters()
    print("플레이할 캐릭터를 고르세요 (1인 1캐릭터):")
    for i, (_, name, cls_ko) in enumerate(chars, 1):
        print(f"  {i}. {name} ({cls_ko})")
    while True:
        try:
            ans = ask("> ").strip().lstrip("﻿")
        except EOFError:
            return None
        if ans.isdigit() and 1 <= int(ans) <= len(chars):
            return chars[int(ans) - 1][0]
        print(f"1~{len(chars)} 중 숫자를 입력하세요.")


def play(engine, narrator, ask=input):
    """게임 반복문. ask는 입력 함수라서, 테스트·play_script는 input 대신 정해진 답을 돌려주는 함수를 넣는다."""
    s = engine.s
    print(f"「잠긴 종」 장면 {s.scene} · {engine.c.scene_titles[s.scene]} — {engine.hero.name}({engine.hero.cls_ko})\n{HELP}\n")
    print_events(engine.opening)
    while True:
        if engine.s.ended == "game_over":
            print(f"\n{LINE}\n게임 오버. r = 체크포인트({engine.checkpoint_label})에서 다시 시작, q = 종료")
            try:
                ans = ask("> ").strip().lstrip("﻿").lower()
            except EOFError:
                break
            if ans == "r":
                print_events(engine.restart_from_checkpoint())
                continue
            break
        if engine.s.ended:
            break
        views = engine.available_choices()
        if not views:
            print("더 할 수 있는 행동이 없습니다.")
            break
        print(f"\n{LINE}")
        for v in views:
            extra = f"  [{v.check_text}]" if v.check_text else ""
            print(f" {v.key:>3}. {v.label}{extra}")
        try:
            ans = ask("> ").strip().lstrip("﻿").lower()
        except EOFError:  # 입력이 끝나면 조용히 종료
            print()
            break
        if ans == "q":
            break
        if ans == "s":
            status(engine)
            continue
        if ans == "k":
            known(engine)
            continue
        if ans in ("h", "?"):
            print(HELP)
            continue
        by_key = {v.key: v for v in views}
        if ans not in by_key:
            print(f"'{ans}'은(는) 지금 고를 수 있는 키가 아닙니다. 목록 왼쪽의 키를 입력하세요. (h = 도움말)")
            continue
        view = by_key[ans]
        actor = pick_actor(engine, view, ask)
        events = engine.choose(view.id, actor)
        print_events(events)
        text = narrator.narrate(engine, events, actor)
        if text:
            print(f"\n  ✦ {text}")
    if engine.s.ended and engine.s.ended != "game_over":
        print(f"\n=== 종료 ({engine.s.ended}) ===")
        status(engine)


def cli_main(argv=None):
    sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser()
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--char", default=None, help="bran / harin / seo / maren (생략하면 시작할 때 선택)")
    p.add_argument("--narrator", choices=["template", "ollama"], default="template")
    p.add_argument("--ollama-url", default="http://localhost:11434")
    p.add_argument("--model", default="qwen2.5:7b")
    p.add_argument("--combat", default=None,
                   help="테스트용: 이 조우로 바로 전투 시작 (rats, shriekers, crabs, ooze, zombies, armor, shadows, octopus, specter, ending2_hold)")
    p.add_argument("--potions", type=int, default=0, help="테스트용: 시작 물약 개수")
    a = p.parse_args(argv)
    narrator = OllamaNarrator(a.ollama_url, a.model) if a.narrator == "ollama" else TemplateNarrator()
    content = Content()
    if a.combat and a.combat not in content.encounters:
        print(f"없는 조우: {a.combat} (가능: {', '.join(content.encounters)})")
        return
    char_id = a.char or choose_character(content, input)
    if char_id is None:
        return
    engine = Engine(content=content, seed=a.seed, char_id=char_id)
    if a.potions:
        engine.s.inventory["potion_healing"] = a.potions
    if a.combat:  # 장면에 아직 연결되지 않은 전투를 직접 해 보기 위한 시험 모드 (W6)
        level = content.encounters[a.combat]["level"]
        engine.opening = (engine._level_up(engine.hero, level) +
                          [engine.opening[0]] + engine.start_combat(a.combat))
    play(engine, narrator)

## [9] 웹 서버 (HTTP JSON API)
원본 파일: `server/webapp.py`

```text
브라우저 게임 화면용 로컬 서버 (표준 라이브러리만 사용).
  python -m server.webapp                 # http://127.0.0.1:8000
  python -m server.webapp --port 8765
  python -m server.webapp --narrator ollama --model qwen2.5:7b
역할 분리:
  - game/      : 규칙·판정·주사위·피해·상태 (기존 W1~W6 엔진, 수정 없이 재사용)
  - server/    : 엔진을 HTTP JSON으로 감싼다. 판정이나 계산을 하지 않는다.
                 화면용 정리(누가 말했는지, 상태 이름 한국어 등)만 한다.
  - web/       : 화면 (HTML·CSS·JS). 서버가 준 상태와 결과를 보여 주기만 한다.
API (모두 JSON, web/app.js의 api()가 부른다):
  GET  /api/meta          캐릭터 4명·조우 목록·장면 제목 (메뉴·캐릭터 선택·전투 시험장 화면용)
  GET  /api/state?sid=    현재 상태만 다시 받기
  POST /api/new           {"char", "seed"?, "combat"?, "potions"?} → 새 Engine을 만들고 세션 id(sid) 발급
  POST /api/choose        {"sid", "id"} → Engine.choose(id)
  POST /api/restart       {"sid"} → Engine.restart_from_checkpoint()
  POST /api/free          {"sid", "text"} → [임시] 자유 문장을 현재 선택지 하나에 연결해 choose
  그 밖의 GET 경로는 web/ 폴더의 정적 파일 (index.html, app.js, assets/…)
응답 형식 (/api/new·choose·restart·free·state): Game.reply() → {"sid", "mode", "events", "narration", "state"}
세션은 메모리(Game.sessions)에만 있어서 서버를 끄면 사라진다 (저장 기능 없음).
```

**들어 있는 것**
- **`speaker_for`** (함수) — 'say' 문장을 누가 말했는지 (대화 창 초상화용). 따옴표가 있는 문장만, 선택지 id 접두어로 판단한다.
- **`Session`** (클래스) — 브라우저 한 판 = Engine 하나. mode는 "story"(이야기) 또는 "arena"(전투 시험장).
- **`Game`** (클래스) — 서버 전체에서 하나. Content(데이터)는 한 번만 읽어 모든 세션이 같이 쓰고, 세션은 sid로 찾는다.
- **`_title`** (함수) — 카드 문장의 첫 마디를 제목으로 (예: '할빅 — 에코 타운의 촌장' → '할빅', '소문 — 종소리.' → '소문 — 종소리').
- **`_resources_ko`** (함수) — 직업 자원(주문 슬롯·Second Wind 등 남은 횟수)을 상태창용 한국어 줄로.
- **`_group`** (함수) — 선택지 id → 화면 구역 이름 (web/app.js GROUP_ORDER·GROUP_LABEL과 짝). 규칙에는 영향 없음.
- **`_plain`** (함수) — Event.data를 JSON으로 보낼 수 있게. 바꿀 수 없는 값이 섞여 있으면 문자열로 바꾼다.
- **`_tokens`** (함수) — 자유 입력 비교용 낱말 집합. 두 글자 이상 낱말만, 끝의 조사는 떼어 낸다.
- **`match_free_text`** (함수) — [임시] 자유 입력 → 현재 선택지 하나. 판정은 엔진이 한다.
- **`make_handler`** (함수) — HTTP 요청 처리 클래스를 만든다. URL 경로를 Game의 메서드로 연결하는 일만 하고,
- **`web_main`** (함수) — (설명 없음)

In [ ]:
WEB = PROJECT_DIR / "web"   # [통합] 원래: 프로젝트 루트/web. 지금: 이 파일이 있는 폴더의 web


MIME = {".html": "text/html; charset=utf-8", ".css": "text/css; charset=utf-8", ".js": "text/javascript; charset=utf-8",
        ".svg": "image/svg+xml", ".png": "image/png", ".json": "application/json; charset=utf-8", ".ico": "image/x-icon"}


CONDITION_KO = {"poisoned": "중독", "antitoxin": "해독제 효과", "restrained": "포박(Restrained)", "grappled": "붙잡힘",
                "dying": "빈사", "stable": "안정(의식 없음)", "dead": "사망", "in_pit": "구덩이에 빠짐",
                "protection_from_evil_and_good": "보호의 결계(집중)", "incapacitated": "행동 불능",
                "frightened": "겁에 질림", "turned": "퇴치됨", "hidden": "숨음"}


FLAG_KO = {
    "got_rina_doll": "리나의 인형을 받았다",
    "opened_chest": "할빅 서재의 서랍을 열었다",
    "heard_lullaby": "메라의 자장가를 끝까지 들었다",
    "well_entrance_known": "우물 입구를 확인했다",
    "bell_chamber_known": "종실의 존재를 알았다",
    "tracked_rats": "쥐 떼의 흔적을 쫓았다",
    "ooze_noticed": "물속의 점액을 알아챘다",
}


VAR_KO = {
    "quest_accepted": "할빅의 의뢰를 받았다",
    "halvik_lie_noticed": "할빅의 말이 어딘가 어긋난다고 느꼈다",
    "ev_body_marks": "할빅의 손에서 수상한 흔적을 보았다",
    "needle_found": "서랍 자물쇠의 독침을 발견했다",
    "well_recently_opened": "우물 뚜껑이 최근에 열린 흔적을 찾았다",
    "rope_anchored": "갱도에 밧줄과 피톤을 고정했다",
    "water_cut_noticed": "측면 물길이 누군가 뚫은 것임을 알아챘다",
    "rats_driven": "측면 통로의 쥐 떼를 몰아냈다",
    "shrieker_resolved": "100피트 지점의 버섯 무리를 지나왔다",
    "shrieker_alerted": "버섯의 비명이 갱도 아래까지 울려 퍼졌다",
}


# 대화 패널에 보여 줄 화자: 원본 NPC 카드 → 초상화 키
NPC_CARD = {"scn-s1-npc-halvik": "halvik", "scn-s1-npc-mera": "mera", "scn-q4-npc-mera-song": "mera",
            "scn-q1-npc-sera": "sera", "scn-m-npc-smith": "gran", "scn-m-npc-innkeeper": "odan",
            "scn-g-rumors": "villagers"}


NPC_NAME = {"halvik": "할빅", "mera": "메라", "sera": "세라", "gran": "그란", "odan": "오단", "villagers": "마을 사람들"}


# 선택지 → 그 행동에서 나온 'say' 문장의 화자 (화면 배치용)
SPEAKER_BY_CHOICE = [("hh_", "halvik"), ("mk_innkeeper", "odan"), ("mk_rumor_halvik", "odan"), ("mk_smith", "gran"),
                     ("buy:rope", "gran"), ("buy:iron_spikes", "gran"), ("ws_mera", "mera"), ("ws_listen", "mera"),
                     ("sq_sera", "sera"), ("sq_take_doll", "sera"), ("move:halvik_study", "halvik")]


def speaker_for(choice_id, text):
    """'say' 문장을 누가 말했는지 (대화 창 초상화용). 따옴표가 있는 문장만, 선택지 id 접두어로 판단한다."""
    if '"' not in text and "“" not in text:
        return None
    for prefix, who in SPEAKER_BY_CHOICE:
        if choice_id and choice_id.startswith(prefix):
            return who
    return None

In [ ]:
class Session:
    """브라우저 한 판 = Engine 하나. mode는 "story"(이야기) 또는 "arena"(전투 시험장).
    lock: 서버가 요청을 여러 스레드로 받으므로, 같은 판에 두 요청이 동시에 들어와도 차례로 처리하게 한다."""

    def __init__(self, engine, narrator):
        self.engine = engine
        self.narrator = narrator
        self.lock = threading.Lock()
        self.mode = "story"

In [ ]:
class Game:
    """서버 전체에서 하나. Content(데이터)는 한 번만 읽어 모든 세션이 같이 쓰고, 세션은 sid로 찾는다.
    narrator_factory: 세션마다 서술기를 새로 만드는 함수 (--narrator 옵션에 따라 TemplateNarrator 또는 Ollama)."""

    def __init__(self, narrator_factory):
        self.content = Content()
        self.sessions = {}
        self.narrator_factory = narrator_factory

    # ------------------------------------------------------------ 메타 정보
    def meta(self):
        """/api/meta. 캐릭터 AC·HP 등은 임시 Engine을 만들어 엔진 계산값을 그대로 쓴다 (서버가 따로 계산하지 않음)."""
        chars = []
        for c in self.content._chars:
            e = Engine(content=self.content, seed=0, char_id=c["id"])
            h = e.hero
            chars.append({
                "id": h.id, "name": h.name, "cls": h.cls, "cls_ko": h.cls_ko, "species": h.species,
                "background": h.background, "hp": h.max_hp, "ac": e.armor_class(h),
                "abilities": [{"id": a, "ko": ABILITY_KO[a], "score": h.score(a), "mod": h.mod(a)} for a in ABILITIES],
                "skills": [SKILL_KO[s] for s in h.skill_profs],
                "weapons": [self.content.rules["weapons"][w]["name"] for w in h.weapons],
                "spells": [self.content.rules["spells"][s]["name"] for s in h.cantrips + h.spells],
                "darkvision": h.darkvision,
            })
        encs = [{"id": k, "name": v["name"], "level": v["level"], "scene": v["scene"],
                 "monsters": [{"kind": g["monster"], "name": self.content.monsters[g["monster"]]["name"], "count": g["count"]}
                              for g in v["monsters"]]}
                for k, v in self.content.encounters.items()]
        return {"characters": chars, "encounters": encs, "save_available": False,
                "scenes": {str(k): {"title": v, "status": self.content.scene_status[k]} for k, v in self.content.scene_titles.items()}}

    # ------------------------------------------------------------ 세션
    def new(self, body):
        """/api/new. 새 Engine(새 게임)을 만들고 sid를 발급한다. combat이 있으면 전투 시험장 모드로 바로 전투 시작."""
        e = Engine(content=self.content, seed=body.get("seed"), char_id=body["char"])
        sess = Session(e, self.narrator_factory())
        events = e.opening
        if body.get("potions"):
            e.s.inventory["potion_healing"] = int(body["potions"])
        if body.get("combat"):  # 전투 시험장: 장면에 아직 연결되지 않은 W6 조우를 바로 시작
            enc = body["combat"]
            if enc not in self.content.encounters:
                raise ValueError(f"없는 조우: {enc}")
            sess.mode = "arena"
            events = e._level_up(e.hero, self.content.encounters[enc]["level"]) + e.start_combat(enc)
        sid = secrets.token_hex(8)
        self.sessions[sid] = sess
        return self.reply(sid, sess, events, None)

    def get(self, sid):
        if sid not in self.sessions:
            raise KeyError("세션이 없습니다. 새 게임을 시작하세요.")
        return self.sessions[sid]

    def choose(self, body):
        """/api/choose. 화면에서 누른 선택지 id를 엔진에 그대로 넘긴다 (지금 고를 수 없는 id면 엔진이 ValueError → 400)."""
        sess = self.get(body["sid"])
        with sess.lock:
            events = sess.engine.choose(body["id"])
            return self.reply(body["sid"], sess, events, body["id"])

    def restart(self, body):
        sess = self.get(body["sid"])
        with sess.lock:
            events = sess.engine.restart_from_checkpoint()
            return self.reply(body["sid"], sess, events, None)

    def free(self, body):
        """[임시] 자유 입력 → 현재 선택지 중 낱말이 가장 많이 겹치는 것을 고른다. 판정은 엔진이 한다."""
        sess = self.get(body["sid"])
        with sess.lock:
            views = sess.engine.available_choices()
            best, score, ranked = match_free_text(body.get("text", ""), views)
            if best is None:
                sugg = [v.label for v in ranked[:3]]
                return dict(self.reply(body["sid"], sess, [], None, narrate=False),
                            free={"matched": None, "suggestions": sugg,
                                  "message": "그 행동은 아직 직접 처리할 수 없습니다. 아래 선택지 중에서 골라 주세요."})
            events = sess.engine.choose(best.id)
            out = self.reply(body["sid"], sess, events, best.id)
            out["free"] = {"matched": best.label, "suggestions": [], "message": f"‘{best.label}’(으)로 처리했습니다."}
            return out

    # ------------------------------------------------------------ 응답 만들기
    def reply(self, sid, sess, events, choice_id, narrate=True):
        """모든 API 응답의 공통 형식. events = 이번 행동의 결과(화면 기록용), state = 행동 뒤 전체 상태(snapshot).
        AI 서술은 TemplateNarrator가 아닐 때만 만든다."""
        e = sess.engine
        evs = [self.event_json(e, ev, choice_id) for ev in events if ev.kind != "flag"]
        narration = None
        if narrate and events and not isinstance(sess.narrator, TemplateNarrator):
            narration = sess.narrator.narrate(e, events, e.hero.id)
        return {"sid": sid, "mode": sess.mode, "events": evs, "narration": narration, "state": self.snapshot(e)}

    def event_json(self, e, ev, choice_id):
        """Event 하나를 JSON으로. 카드면 카드 종류·라벨을, NPC 문장이면 화자(초상화 키)를 덧붙인다."""
        out = {"kind": ev.kind, "text": ev.text, "data": _plain(ev.data)}
        if ev.kind == "card":
            card = self.content.cards[ev.data["id"]]
            out["card_kind"] = card["kind"]
            out["label"] = card["label"]
            if ev.data["id"] in NPC_CARD:
                who = NPC_CARD[ev.data["id"]]
                out["speaker"], out["speaker_name"] = who, NPC_NAME[who]
        elif ev.kind == "say":
            who = speaker_for(choice_id, ev.text)
            if who:
                out["speaker"], out["speaker_name"] = who, NPC_NAME[who]
        return out

    def snapshot(self, e):
        """화면이 그리는 상태 전체 (상단 장소, 상태창 4개 탭, 전투 패널, 선택지 버튼). web/app.js renderState()가 읽는다.

        숨김 원칙: 단서 탭에는 플레이어에게 이미 보여 준 카드(player_known_cards)만 넣고,
        적은 HP 숫자 대신 wound(멀쩡함/다침/피투성이)만 보낸다. 선택지는 group으로 묶어 화면이 구역별로 나눈다.
        """
        s, h, c = e.s, e.hero, self.content
        clues = []
        for cid in e.player_known_cards(h.id):
            card = c.cards[cid]
            if card["kind"] in ("clue", "rumor", "npc"):
                text = e._view_text(cid)[0]
                clues.append({"id": cid, "kind": card["kind"], "label": _title(text, card["label"]), "text": text,
                              "room": c.room_name(card["room"]) if card["room"] else ""})
        progress = [FLAG_KO.get(f, f) for f in sorted(s.flags)]
        progress += [VAR_KO[k] for k in VAR_KO if s.vars.get(k)]
        combat = None
        if s.combat:
            cb = s.combat
            enemies = []
            for m in cb["enemies"].values():
                md = c.monsters[m["kind"]]
                enemies.append({"id": m["id"], "kind": m["kind"], "name": m["name"], "wound": e.wound_text(m) if not m["out"] else
                                ("쓰러짐" if m["out"] == "dead" else "달아남"), "out": m["out"], "ac": md["ac"], "type": md["type"],
                                "conditions": [CONDITION_KO.get(k, k) for k in m["conditions"]],
                                "bloodied": m["hp"] * 2 <= m["hp_max"]})
            order = [h.name if o == "hero" else cb["enemies"][o]["name"] for o in cb["order"]]
            combat = {"encounter": cb["enc"], "name": c.encounters[cb["enc"]]["name"], "round": cb["round"],
                      "enemies": enemies, "order": order, "turn_index": cb["idx"],
                      "turn": cb["turn"], "dodging": cb["dodging"]}
        choices = []
        for v in e.available_choices():
            group = _group(v.id)
            choices.append({"id": v.id, "key": v.key, "label": v.label, "check": v.check_text, "group": group})
        inv = [{"id": i, "name": c.items[i]["name"], "count": n, "quest": bool(c.items[i].get("quest"))}
               for i, n in s.inventory.items() if n]
        room = c.encounters[s.combat["enc"]]["room"] if s.combat else s.room   # 전투 시험장: 조우 장소 표시
        return {
            "scene": s.scene, "scene_title": c.scene_titles[s.scene], "scene_status": c.scene_status.get(s.scene),
            "room": room, "room_name": c.room_name(room), "minutes": s.minutes,
            "gold": fmt_money(s.gold_cp), "ended": s.ended, "checkpoint": e.checkpoint_label,
            "hero": {
                "id": h.id, "name": h.name, "cls": h.cls, "cls_ko": h.cls_ko, "species": h.species, "level": h.level,
                "hp": h.hp, "max_hp": h.max_hp, "ac": e.armor_class(h),
                "abilities": [{"id": a, "ko": ABILITY_KO[a], "score": h.score(a), "mod": h.mod(a)} for a in ABILITIES],
                "conditions": [CONDITION_KO.get(k, k) for k in h.conditions],
                "hd": f"{h.hd_left}/{h.hd_max} (d{h.hit_die})", "inspiration": h.heroic_inspiration,
                "resources": _resources_ko(h.resources), "down": h.down,
                "death_saves": h.conditions.get("dying") if isinstance(h.conditions.get("dying"), dict) else None,
            },
            "inventory": inv, "clues": clues, "progress": progress, "visited": [c.room_name(r) for r in sorted(s.visited)],
            "combat": combat, "choices": choices, "pending": (s.pending or {}).get("type"),
        }

In [ ]:
def _title(text, fallback):
    """카드 문장의 첫 마디를 제목으로 (예: '할빅 — 에코 타운의 촌장' → '할빅', '소문 — 종소리.' → '소문 — 종소리')."""
    head = re.split(r"[.。]", text or "", maxsplit=1)[0]
    if head.startswith("소문 —"):
        return head[:24]
    head = head.split(" — ")[0]
    return head[:24] if head else fallback


def _resources_ko(r):
    """직업 자원(주문 슬롯·Second Wind 등 남은 횟수)을 상태창용 한국어 줄로."""
    names = {"second_wind": "Second Wind", "action_surge": "Action Surge", "channel_divinity": "Channel Divinity",
             "arcane_recovery": "Arcane Recovery"}
    out = []
    for k, v in r.items():
        if k == "slots":
            out.append(f"주문 슬롯 " + " / ".join(f"{i + 1}레벨 {n}" for i, n in enumerate(v)))
        else:
            out.append(f"{names.get(k, k)} {v}회")
    return out


def _group(cid):
    """선택지 id → 화면 구역 이름 (web/app.js GROUP_ORDER·GROUP_LABEL과 짝). 규칙에는 영향 없음."""
    kind = cid.split(":")[0]
    if kind == "cmb":
        sub = cid.split(":")[1]
        if sub in ("use", "wind") or cid.endswith(":healing_word:self"):
            return "bonus"
        if sub in ("option", "end", "surge"):
            return "combat_other"
        return "combat_action"
    return {"move": "move", "buy": "shop", "use": "item", "rest": "rest", "revisit": "action",
            "inspire": "pending", "hd": "pending", "wake": "pending", "kit": "pending", "aid": "pending"}.get(kind, "action")


def _plain(d):
    """Event.data를 JSON으로 보낼 수 있게. 바꿀 수 없는 값이 섞여 있으면 문자열로 바꾼다."""
    try:
        json.dumps(d)
        return d
    except TypeError:
        return {k: str(v) for k, v in d.items()}


TOKEN = re.compile(r"[가-힣A-Za-z0-9']+")


JOSA = re.compile(r"(에게서|에게|에서|으로|을|를|이|가|은|는|에|로|와|과|의|도|만)$")


MOVE_WORDS = ("간다", "가자", "가다", "가본다", "이동", "향한다", "들어간다", "나간다", "돌아간다")


def _tokens(text):
    """자유 입력 비교용 낱말 집합. 두 글자 이상 낱말만, 끝의 조사는 떼어 낸다."""
    out = set()
    for t in TOKEN.findall(text or ""):
        while len(t) > 2:                      # 조사를 떼어 낸다 (우물가로 → 우물가 → 우물)
            s = JOSA.sub("", t)
            if s == t:
                break
            t = s
        if len(t) >= 2:
            out.add(t)
    return out


def match_free_text(text, views):
    """[임시] 자유 입력 → 현재 선택지 하나. 판정은 엔진이 한다.
    점수: 겹치는 낱말마다 (2 ÷ 그 낱말이 들어 있는 선택지 수), 동사(…다/…고)는 절반, 부분 일치는 다시 절반.
    이동하는 말이 있으면 이동 선택지에 가산. 1위가 1.5점 이상이고 2위보다 0.5점 넘게 높을 때만 처리."""
    want = _tokens(text)
    labels = [(v, _tokens(v.label)) for v in views]
    df = {w: max(1, sum(1 for _, ls in labels if any(w in l or l in w for l in ls))) for w in want}
    moving = any(m in (text or "") for m in MOVE_WORDS)
    scored = []
    for v, ls in labels:
        s = 0.0
        for w in want:
            best = 0.0
            for l in ls:
                if w in l or l in w:
                    weight = 2 / df[w] * (0.5 if w.endswith(("다", "고")) else 1)
                    best = max(best, weight if w == l else weight / 2)
            s += best
        if moving and v.id.startswith("move:") and s > 0:
            s += 1.5
        scored.append((s, v))
    scored.sort(key=lambda x: -x[0])
    ranked = [v for s, v in scored if s > 0] or [v for _, v in scored]
    if not scored or scored[0][0] < 1.5 or (len(scored) > 1 and scored[1][0] > scored[0][0] - 0.5):
        return None, 0, ranked
    return scored[0][1], scored[0][0], ranked


def make_handler(game):
    """HTTP 요청 처리 클래스를 만든다. URL 경로를 Game의 메서드로 연결하는 일만 하고,
    오류는 KeyError → 404(세션 없음 등), ValueError·RuntimeError → 400(고를 수 없는 선택지 등)으로 돌려준다."""
    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def _send(self, code, body, ctype="application/json; charset=utf-8"):
            data = body if isinstance(body, bytes) else json.dumps(body, ensure_ascii=False).encode("utf-8")
            self.send_response(code)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(data)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            url = urlparse(self.path)
            if url.path == "/api/meta":
                return self._send(200, game.meta())
            if url.path == "/api/state":
                sid = parse_qs(url.query).get("sid", [""])[0]
                try:
                    sess = game.get(sid)
                except KeyError as err:
                    return self._send(404, {"error": str(err)})
                return self._send(200, game.reply(sid, sess, [], None, narrate=False))
            # 정적 파일: web/ 폴더 밖을 가리키는 경로(../ 등)는 404로 막는다
            path = (WEB / url.path.lstrip("/")).resolve() if url.path != "/" else WEB / "index.html"
            if not str(path).startswith(str(WEB.resolve())) or not path.is_file():
                return self._send(404, {"error": "not found"})
            return self._send(200, path.read_bytes(), MIME.get(path.suffix, "application/octet-stream"))

        def do_POST(self):
            url = urlparse(self.path)
            try:
                body = json.loads(self.rfile.read(int(self.headers.get("Content-Length") or 0)) or b"{}")
                route = {"/api/new": game.new, "/api/choose": game.choose, "/api/restart": game.restart,
                         "/api/free": game.free}.get(url.path)
                if not route:
                    return self._send(404, {"error": "not found"})
                return self._send(200, route(body))
            except KeyError as err:
                return self._send(404, {"error": str(err)})
            except (ValueError, RuntimeError) as err:
                return self._send(400, {"error": str(err)})

    return Handler


def web_main(argv=None):
    sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser()
    p.add_argument("--host", default="127.0.0.1")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--narrator", choices=["template", "ollama"], default="template")
    p.add_argument("--ollama-url", default="http://localhost:11434")
    p.add_argument("--model", default="qwen2.5:7b")
    a = p.parse_args(argv)
    factory = (lambda: OllamaNarrator(a.ollama_url, a.model)) if a.narrator == "ollama" else TemplateNarrator
    ThreadingHTTPServer.allow_reuse_address = False   # Windows에서 같은 포트에 서버가 두 개 뜨지 않게 (세션이 섞임)
    try:
        srv = ThreadingHTTPServer((a.host, a.port), make_handler(Game(factory)))
    except OSError:
        print(f"포트 {a.port}을(를) 이미 다른 프로그램(이미 켜 둔 게임 서버 등)이 쓰고 있습니다. "
              f"켜져 있는 서버 창을 쓰거나, 끄고 다시 실행하세요. 다른 포트: --port 8765", flush=True)
        return
    print(f"「잠긴 종」 게임 서버: http://{a.host}:{a.port}  (종료: Ctrl+C)", flush=True)
    try:
        srv.serve_forever()
    except KeyboardInterrupt:
        pass

## [10] 도구: 1인 전투 난이도 측정
원본 파일: `tools/combat_sim.py`

```text
1인 전투 측정 도구 (W6, 05-6 5-5 2단계). 엔진의 실제 전투 코드를 그대로 돌린다.
  python tools/combat_sim.py                   # 모든 조우 × 4직업 × 200회, 완화 수단 미사용/사용
  python tools/combat_sim.py --runs 500 --enc shadows --out analysis/sim.md
행동 규칙 (단순한 기본 전술, [보완]):
  1) 완화 수단 사용 모드에서 HP가 50% 이하이면 원본 완화 선택지(물러나기·횃불 끄기 등)를 고름
  2) HP가 40% 이하이면 회복 (물약·Second Wind·Healing Word·Cure Wounds 순)
  3) Restrained로 붙잡혔으면 탈출 시도
  4) 언데드가 2마리 이상이면 Turn Undead
  5) 남은 HP가 가장 적은 적에게 기대 피해가 가장 큰 공격·주문
판정은 모두 엔진이 한다. 이 도구는 '어떤 선택지를 고를지'만 정한다.
```

**들어 있는 것**
- **`avg`** (함수) — (설명 없음)
- **`p_hit`** (함수) — (설명 없음)
- **`mult`** (함수) — (설명 없음)
- **`expected`** (함수) — 선택지의 기대 피해 (대략). 행동 고르기용.
- **`pick`** (함수) — 위 머리말의 행동 규칙 1)~5)대로 선택지 id 하나를 고른다 (사람 대신 누르는 역할).
- **`run_one`** (함수) — 전투 1회를 끝까지. (결과, 남은 HP, 최대 HP). 결과: victory/retreat/avoid/peace/death/defeat/unfinished.
- **`measure`** (함수) — 조우 × 캐릭터 4명 × 완화 수단(미사용/사용) 조합마다 runs번 돌려 결과를 센다 (시드 0~runs-1).
- **`to_markdown`** (함수) — (설명 없음)
- **`sim_main`** (함수) — (설명 없음)

In [ ]:
CHARS = ["bran", "harin", "seo", "maren"]


HEALS = ["cmb:use:potion_healing", "cmb:wind", "cmb:cast:healing_word:self", "cmb:cast:cure_wounds:self"]


def avg(expr):
    m = DICE_EXPR.match(expr)
    n, s = int(m.group(1) or 1), int(m.group(2))
    b = int(m.group(3).replace(" ", "")) if m.group(3) else 0
    return n * (s + 1) / 2 + b


def p_hit(bonus, ac):
    return min(0.95, max(0.05, (21 - (ac - bonus)) / 20))


def mult(md, dtype):
    if dtype in md["immune"]:
        return 0
    if dtype in md["resist"]:
        return 0.5
    if dtype in md["vulnerable"]:
        return 2
    return 1


def expected(e, cid):
    """선택지의 기대 피해 (대략). 행동 고르기용."""
    parts = cid.split(":")
    hero, cb = e.hero, e.s.combat
    if parts[1] == "attack":
        m = cb["enemies"][parts[3]]
        md = e.c.monsters[m["kind"]]
        w = e.c.rules["weapons"][parts[2]]
        ab, bonus = e.weapon_attack(hero, parts[2])
        return p_hit(bonus, md["ac"]) * (avg(w["damage"]) + hero.mod(ab)) * mult(md, w["type"])
    if parts[1] == "cast" and parts[3] != "self":
        sp = e.c.rules["spells"][parts[2]]
        m = cb["enemies"][parts[3]]
        md = e.c.monsters[m["kind"]]
        if sp["kind"] == "attack":
            return p_hit(e.spell_attack_bonus(hero), md["ac"]) * avg(sp["damage"]) * mult(md, sp["type"])
        if sp["kind"] == "save":
            pf = min(0.95, max(0.05, (e.spell_dc(hero) - 1 - md["saves"][sp["save"]]) / 20))
            return pf * avg(sp["damage"]) * mult(md, sp["type"])
        if sp["kind"] == "missile":
            return sp["darts"] * avg(sp["damage"]) * mult(md, sp["type"])
    if parts[1] == "holy":
        m = cb["enemies"][parts[2]]
        md = e.c.monsters[m["kind"]]
        if md["type"] not in ("undead", "fiend"):
            return 0
        dc = 8 + hero.mod("dex") + hero.proficiency_bonus
        pf = min(0.95, max(0.05, (dc - 1 - md["saves"]["dex"]) / 20))
        return pf * 9 * mult(md, "radiant")
    return -1


def pick(e, views, mitigate):
    """위 머리말의 행동 규칙 1)~5)대로 선택지 id 하나를 고른다 (사람 대신 누르는 역할)."""
    hero, cb = e.hero, e.s.combat
    ids = {v.id: v for v in views}
    opts = [v.id for v in views if v.id.startswith("cmb:option:")]
    if mitigate and opts and hero.hp * 2 <= hero.max_hp:   # 완화 수단이 회복보다 먼저
        return opts[0]
    if hero.hp * 10 <= hero.max_hp * 4:
        for h in HEALS:
            if h in ids:
                return h
    if "cmb:escape" in ids and "restrained" in hero.conditions:
        return "cmb:escape"
    undead = [m for m in e.living_enemies() if e.c.monsters[m["kind"]]["type"] == "undead"
              and "turned" not in m["conditions"]]
    if "cmb:turn" in ids and len(undead) >= 2:
        return "cmb:turn"
    if "cmb:surge" in ids:
        return "cmb:surge"
    foes = sorted(e.living_enemies(), key=lambda m: (m["hp"], m["id"]))
    for target in foes:
        if "turned" in target["conditions"] and len(foes) > 1:
            continue  # 겁먹은 적은 나중에 (피해를 주면 풀림)
        acts = [v.id for v in views if v.id.endswith(":" + target["id"])]
        if acts:
            best = max(acts, key=lambda c: expected(e, c))
            if expected(e, best) > 0:
                return best
    if "cmb:dodge" in ids and e.s.combat["turn"]["action"] > 0:
        return "cmb:dodge"
    return "cmb:end"


def run_one(enc_id, char_id, seed, mitigate, potions=0):
    """전투 1회를 끝까지. (결과, 남은 HP, 최대 HP). 결과: victory/retreat/avoid/peace/death/defeat/unfinished."""
    e = Engine(seed=seed, char_id=char_id)
    lvl = e.c.encounters[enc_id]["level"]
    e._level_up(e.hero, lvl)
    if potions:
        e.s.inventory["potion_healing"] = potions
    e.start_combat(enc_id)
    for _ in range(2000):
        if not e.s.combat or e.s.ended:
            break
        e.choose(pick(e, e.available_choices(), mitigate))
    if "dead" in e.hero.conditions:
        result = "death"
    elif e.s.ended == "game_over":
        result = "defeat"
    else:
        result = e.s.vars.get(f"combat:{enc_id}", "unfinished")
    return result, e.hero.hp, e.hero.max_hp


def measure(enc_ids, runs, potions=0):
    """조우 × 캐릭터 4명 × 완화 수단(미사용/사용) 조합마다 runs번 돌려 결과를 센다 (시드 0~runs-1)."""
    rows = []
    for enc_id in enc_ids:
        for char_id in CHARS:
            for mitigate in (False, True):
                c = Counter()
                hp_left = 0
                for seed in range(runs):
                    res, hp, mx = run_one(enc_id, char_id, seed, mitigate, potions)
                    c[res] += 1
                    hp_left += hp / mx if mx else 0
                rows.append((enc_id, char_id, mitigate, c, hp_left / runs))
    return rows


def to_markdown(rows, runs, potions):
    lines = [f"| 조우 | 캐릭터 | 완화 수단 | 승리 | 물러남·회피 | 사망 | 1인 패배 | 게임 오버 합계 | 남은 HP 평균 |",
             "|---|---|---|---|---|---|---|---|---|"]
    for enc_id, char_id, mit, c, hp in rows:
        pct = lambda k: 100 * c[k] / runs  # noqa: E731
        go = pct("death") + pct("defeat")
        lines.append(f"| {enc_id} | {char_id} | {'사용' if mit else '미사용'} | {pct('victory'):.0f}% | "
                     f"{pct('retreat') + pct('avoid') + pct('peace'):.0f}% | {pct('death'):.0f}% | {pct('defeat'):.0f}% | "
                     f"**{go:.0f}%** | {hp:.0%} |")
    head = f"(각 칸 {runs}회, 시드 0~{runs - 1}, 시작 물약 {potions}개, 전투 1회만 단독 측정)\n\n"
    return head + "\n".join(lines)


def sim_main(argv=None):
    sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser()
    p.add_argument("--runs", type=int, default=200)
    p.add_argument("--enc", nargs="*", default=None)
    p.add_argument("--potions", type=int, default=0)
    p.add_argument("--out", default=None)
    a = p.parse_args(argv)
    e = Engine(seed=0, char_id="bran")
    enc_ids = a.enc or list(e.c.encounters)
    md = to_markdown(measure(enc_ids, a.runs, a.potions), a.runs, a.potions)
    print(md)
    if a.out:
        Path(a.out).write_text(md + "\n", encoding="utf-8")

## [11] 도구: 정해진 경로로 터미널 게임 자동 진행
원본 파일: `tools/play_script.py`

```text
정해진 행동 순서로 터미널 게임(game.cli.play)을 끝까지 실행해 출력 전체를 확인한다.
사람이 번호를 누르는 대신, 각 단계에서 '보여진 선택지 목록에서 이름(id)으로 찾아' 그 번호를 입력한다.
  python tools/play_script.py [--seed N]
```

**들어 있는 것**
- **`play_script_main`** (함수) — (설명 없음)

In [ ]:
ROUTE = [
    "sq_rumors", "sq_sera", "sq_take_doll", "move:halvik_house", "hh_meet", "hh_ask_missing", "hh_ask_bell",
    "hh_crest", "hh_hands", "hh_hands_alt", "hh_bellcord", "hh_ask_key", "move:halvik_study",
    "st_detect_poison", "st_inspect_lock", "st_disarm",
    "st_pick_lock", "st_use_key", "move:halvik_house", "move:village_square", "move:village_market",
    "mk_innkeeper", "mk_rumor_halvik", "mk_goods", "buy:rope", "buy:iron_spikes", "mk_rest",
    "move:village_square", "move:well_side", "ws_mera", "ws_listen_ask", "ws_listen_wait",
    "ws_rumor_monks", "ws_open_well", "ws_well_traces", "ws_pulley", "s", "ws_descend",
]


def play_script_main():
    sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser()
    p.add_argument("--seed", type=int, default=11)
    p.add_argument("--char", default="harin")
    a = p.parse_args()
    engine = Engine(seed=a.seed, char_id=a.char)
    route = list(ROUTE)

    def ask(prompt):
        if prompt.strip().startswith("누가"):
            return ""  # 추천 캐릭터
        if engine.s.ended == "game_over":
            print("r   ← 체크포인트에서 다시 시작")
            return "r"
        views = engine.available_choices()
        while route:
            want = route.pop(0)
            if want == "s":
                print("s")
                return "s"
            for v in views:
                if v.id == want:
                    print(f"{v.key}   ← {want}")
                    return v.key
            print(f"(건너뜀: {want} — 지금 고를 수 없음)")
        return "q"

    play(engine, TemplateNarrator(), ask=ask)

## [공유용 추가] 실행 메뉴와 예시 함수

```text
원본의 실행 방법과 대응: python -m game.cli → cli / python -m server.webapp → web
                             python tools/combat_sim.py → sim / python tools/play_script.py → playscript
    demo와 check는 팀 공유용으로 새로 넣은 짧은 예시이며, 게임 규칙에는 손대지 않는다.
```

**들어 있는 것**
- **`share_check`** (함수) — 필요한 외부 파일이 있는지 확인만 한다 (아무것도 만들거나 고치지 않음). 모두 있으면 True.
- **`share_demo`** (함수) — 입력 없이 장면 1 앞부분을 진행해 보여 준다: 시작 → 선택지 4개 실행 → 상태 요약.
- **`_share_menu`** (함수) — 명령줄 실행 메뉴. 한국어가 깨지지 않게 출력 인코딩을 UTF-8로 맞춘다 (지원하는 환경에서만).

In [ ]:
REQUIRED_DATA = ["cards.json", "scene1.json", "scene2.json", "characters.json", "items.json",
                 "traps.json", "monsters.json", "encounters.json", "rules.json"]


WEB_FILES = ["index.html", "app.js", "audio.js", "style.css"]


def share_check():
    """필요한 외부 파일이 있는지 확인만 한다 (아무것도 만들거나 고치지 않음). 모두 있으면 True."""
    ok = True
    print(f"기준 폴더: {PROJECT_DIR}")
    for name in REQUIRED_DATA:
        p = DATA / name
        print(f"  [{'있음' if p.is_file() else '없음'}] game/data/{name}")
        ok = ok and p.is_file()
    web_ok = all((WEB / n).is_file() for n in WEB_FILES)
    print(f"  [{'있음' if web_ok else '없음'}] web/ ({', '.join(WEB_FILES)}) — 웹 화면(web 명령)에만 필요")
    print("엔진 실행 가능" if ok else "game/data 파일이 없어 엔진을 만들 수 없습니다.")
    return ok


def share_demo(char_id="harin", seed=7):
    """입력 없이 장면 1 앞부분을 진행해 보여 준다: 시작 → 선택지 4개 실행 → 상태 요약.
    모든 처리는 Engine.choose()가 하고, 이 함수는 고를 id를 정해 줄 뿐이다.
    Colab 노트북 셀에서 import해 바로 불러도 된다."""
    e = Engine(seed=seed, char_id=char_id)
    print(f"== 새 게임: {e.hero.name}({e.hero.cls_ko}), 주사위 시드 {seed} ==")
    print_events(e.opening)
    for cid in ["sq_sera", "sq_take_doll", "move:halvik_house", "hh_meet"]:
        print(f"\n▶ {cid}")
        print_events(e.choose(cid))
    print("\n지금 고를 수 있는 선택지:")
    for v in e.available_choices():
        print(f"  {v.key:>3}. {v.label}" + (f"  [{v.check_text}]" if v.check_text else ""))
    print()
    status(e)


def _share_menu(argv=None):
    """명령줄 실행 메뉴. 한국어가 깨지지 않게 출력 인코딩을 UTF-8로 맞춘다 (지원하는 환경에서만)."""
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    argv = list(sys.argv[1:] if argv is None else argv)
    usage = ("사용법: python team_share_game.py <명령> [옵션]\n"
             "  check       필요한 데이터 파일 확인\n"
             "  demo        입력 없이 장면 1 앞부분 자동 진행\n"
             "  cli  [...]  터미널 플레이 (옵션: --char, --seed, --combat, --potions, --narrator)\n"
             "  web  [...]  웹 서버 (옵션: --port, --host, --narrator)\n"
             "  sim  [...]  전투 측정 (옵션: --runs, --enc, --potions, --out)\n"
             "  playscript  정해진 경로로 장면 1 자동 진행 (옵션: --seed, --char)")
    if not argv or argv[0] in ("-h", "--help", "help"):
        print(usage)
        return
    cmd, rest = argv[0], argv[1:]
    if cmd == "check":
        share_check()
    elif cmd == "demo":
        share_demo()
    elif cmd == "cli":
        cli_main(rest)
    elif cmd == "web":
        web_main(rest)
    elif cmd == "sim":
        sim_main(rest)
    elif cmd == "playscript":
        sys.argv = [sys.argv[0]] + rest      # 원본 play_script는 sys.argv에서 옵션을 읽는다
        play_script_main()
    else:
        print(f"알 수 없는 명령: {cmd}\n{usage}")

## 사용 예시 (실행하려면 game/data 필요)
위 셀을 모두 실행한 뒤 아래 셀을 실행하면 장면 1 앞부분이 자동으로 진행됩니다. 직접 플레이는 `play(Engine(char_id="harin"), TemplateNarrator())` (끝내려면 q).

In [ ]:
share_check()
# share_demo()                                            # 장면 1 앞부분 자동 진행
# play(Engine(char_id="harin"), TemplateNarrator())      # 직접 플레이 (입력 칸에 키 입력, 끝내려면 q)